In [ ]:
!pip uninstall -y torchao 2>/dev/null || true
!pip install -q --upgrade \
    transformers \
    peft \
    bitsandbytes \
    datasets \
    accelerate \
    scikit-learn \
    "pandas==2.2.2" \
    "numpy<2.1" \
    tqdm

print('Installation complete.')


In [ ]:
import pandas as pd
import numpy as np
print(f'pandas : {pd.__version__}')
print(f'numpy  : {np.__version__}')

In [ ]:
import time
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
import bitsandbytes as bnb
from scipy.stats import t

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    TrainerState,
    TrainerControl,
    set_seed,
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    prepare_model_for_kbit_training,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
)

print('All imports OK.')
print(f'bitsandbytes : {bnb.__version__}')

In [ ]:

file_path = "/content/balanced-reviews.txt"

df = pd.read_csv(
    file_path,
    sep="\t",
    encoding="utf-16"
)

df.head()

In [ ]:
df = df[df["rating"].isin([1, 2, 4, 5])].copy()

df["label"] = df["rating"].replace({
    1: 0,   # Negative
    2: 0,   # Negative
    4: 1,   # Positive
    5: 1    # Positive
})

print(df["label"].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split

X = df["review"]
y = df["label"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print(f"Training size: {len(X_train)}")
print(f"Validation size: {len(X_val)}")
print(f"Test size: {len(X_test)}")

In [ ]:
train_df = pd.DataFrame({
    "review": X_train,
    "label": y_train
})

val_df = pd.DataFrame({
    "review": X_val,
    "label": y_val
})

test_df = pd.DataFrame({
    "review": X_test,
    "label": y_test
})

In [ ]:
def class_stats(df, name):
    counts = df["label"].value_counts().sort_index()
    perc = df["label"].value_counts(normalize=True).sort_index() * 100

    return pd.DataFrame({
        "Split": name,
        "Class": counts.index,
        "Count": counts.values,
        "Percentage (%)": perc.round(2).values
    })

stats = pd.concat([
    class_stats(train_df, "Train"),
    class_stats(val_df, "Validation"),
    class_stats(test_df, "Test")
], ignore_index=True)

stats

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

LOCAL_DIR = '/content/camelbert_local'

DRIVE_DIR = '/content/drive/MyDrive/CAMeLBERT_Study_5Seeds'

EXPERIMENT_ID = 'camelbert_5seeds_final_epoch_v2'

RECOVERY_ROOT = os.path.join(
    DRIVE_DIR,
    EXPERIMENT_ID
)

RESULTS_DIR = os.path.join(
    RECOVERY_ROOT,
    'results'
)

os.makedirs(LOCAL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print('LOCAL_DIR      :', LOCAL_DIR)
print('DRIVE_DIR      :', DRIVE_DIR)
print('EXPERIMENT_ID  :', EXPERIMENT_ID)
print('RECOVERY_ROOT  :', RECOVERY_ROOT)
print('RESULTS_DIR    :', RESULTS_DIR)


In [ ]:
import transformers
import peft

print('=' * 60)
print('ENVIRONMENT SNAPSHOT')
print('=' * 60)
print(f'PyTorch       : {torch.__version__}')
print(f'Transformers  : {transformers.__version__}')
print(f'PEFT          : {peft.__version__}')
print(f'bitsandbytes  : {bnb.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU           : {torch.cuda.get_device_name(0)}')
    total_vram = (
        torch.cuda.get_device_properties(0).total_memory / 1e9
    )
    print(f'VRAM total    : {total_vram:.1f} GB')
else:
    raise RuntimeError(
        'A CUDA GPU is required. Enable GPU runtime in Colab.'
    )

print('=' * 60)

snapshot = {
    'pytorch': torch.__version__,
    'transformers': transformers.__version__,
    'peft': peft.__version__,
    'bitsandbytes': bnb.__version__,
    'cuda': str(torch.cuda.is_available()),
    'gpu': (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else 'N/A'
    ),
    'vram_gb': (
        round(total_vram, 1)
        if torch.cuda.is_available()
        else 'N/A'
    ),
}

snapshot_path = os.path.join(
    RESULTS_DIR,
    'environment_snapshot.json'
)

with open(snapshot_path, 'w') as f:
    json.dump(snapshot, f, indent=2)

print(f'Environment snapshot saved → {snapshot_path}')

In [ ]:


MODEL_NAME = 'CAMeL-Lab/bert-base-arabic-camelbert-mix'
NUM_LABELS = 2

BASE_SPLIT_SEED = 42

SEEDS = [42, 123, 456, 789, 2024]

NUM_EPOCHS    = 10
TRAIN_BATCH   = 32
EVAL_BATCH    = 64
LEARNING_RATE = 2e-5
WARMUP_RATIO  = 0.1
WEIGHT_DECAY  = 0.01

LOAD_BEST_MODEL_AT_END = False

METHODS = [
    'full_finetuning',
    'frozen_backbone',
    'lora',
    'qlora',
]


DATA_LEVELS = {
    'full_dataset': None,
    '25_per_class': 25,
    '100_per_class': 100,
}

LORA_R              = 16
LORA_ALPHA          = 32
LORA_DROPOUT        = 0.1
LORA_TARGET_MODULES = ['query', 'value']

QUANT_TYPE    = 'nf4'
DOUBLE_QUANT  = False
COMPUTE_DTYPE = torch.float16

MAX_SEQ_LENGTH = None


print('=' * 60)
print('GLOBAL EXPERIMENTAL CONFIGURATION')
print('=' * 60)

print(f'Model              : {MODEL_NAME}')
print(f'Methods            : {METHODS}')
print(f'Data levels        : {list(DATA_LEVELS.keys())}')
print(f'Seeds              : {SEEDS}')
print(f'Number of seeds    : {len(SEEDS)}')

print(
    f'Runs per method    : '
    f'{len(DATA_LEVELS) * len(SEEDS)}'
)

print(
    f'Total study runs   : '
    f'{len(METHODS) * len(DATA_LEVELS) * len(SEEDS)}'
)

print(f'Epochs             : {NUM_EPOCHS}')
print(f'Train batch size   : {TRAIN_BATCH}')
print(f'Evaluation batch   : {EVAL_BATCH}')
print(f'Learning rate      : {LEARNING_RATE}')
print(f'Warmup ratio       : {WARMUP_RATIO}')
print(f'Weight decay       : {WEIGHT_DECAY}')
print(f'Load best model    : {LOAD_BEST_MODEL_AT_END}')
print('Final test model   : last training epoch')

print(
    f'LoRA               : '
    f'r={LORA_R}, '
    f'alpha={LORA_ALPHA}, '
    f'dropout={LORA_DROPOUT}'
)

print(
    f'QLoRA              : '
    f'4-bit {QUANT_TYPE.upper()}, '
    f'double quantization={DOUBLE_QUANT}'
)

print('=' * 60)

In [ ]:


def create_nested_reduced_data(train_df, seed):


    required_columns = {'review', 'label'}

    if not required_columns.issubset(train_df.columns):
        raise ValueError(
            f'train_df must contain {required_columns}. '
            f'Found: {train_df.columns.tolist()}'
        )

    labels = sorted(
        train_df['label']
        .dropna()
        .unique()
        .tolist()
    )

    if labels != [0, 1]:
        raise ValueError(
            f'Expected labels [0, 1], but found {labels}.'
        )

    parts_25 = []
    parts_100 = []

    for label in labels:

        class_data = train_df[
            train_df['label'] == label
        ]

        if len(class_data) < 100:
            raise ValueError(
                f'Class {label} contains only '
                f'{len(class_data)} samples.'
            )

        selected_100 = class_data.sample(
            n=100,
            replace=False,
            random_state=seed
        )

        selected_25 = selected_100.sample(
            n=25,
            replace=False,
            random_state=seed
        )

        assert set(selected_25.index).issubset(
            set(selected_100.index)
        ), (
            f'Nesting failed for class {label} '
            f'and seed {seed}.'
        )

        parts_25.append(selected_25)
        parts_100.append(selected_100)

    reduced_25 = (
        pd.concat(parts_25)
        .sample(frac=1, random_state=seed)
        .reset_index(drop=True)
    )

    reduced_100 = (
        pd.concat(parts_100)
        .sample(frac=1, random_state=seed)
        .reset_index(drop=True)
    )

    return reduced_25, reduced_100


REDUCED_DATA_BY_SEED = {}

for seed in SEEDS:

    reduced_25, reduced_100 = (
        create_nested_reduced_data(
            train_df=train_df,
            seed=seed
        )
    )

    REDUCED_DATA_BY_SEED[seed] = {
        '25_per_class': reduced_25,
        '100_per_class': reduced_100,
    }

    print(
        f'Seed {seed}: '
        f'25/class={len(reduced_25)} samples | '
        f'100/class={len(reduced_100)} samples'
    )

print(
    'Seed-specific nested reduced-data subsets '
    'created successfully.'
)

In [ ]:


def compute_metrics(eval_pred):


    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        'accuracy': accuracy_score(
            labels,
            predictions
        ),

        'f1_macro': f1_score(
            labels,
            predictions,
            labels=[0, 1],
            average='macro',
            zero_division=0
        ),

        'precision': precision_score(
            labels,
            predictions,
            labels=[0, 1],
            average='macro',
            zero_division=0
        ),

        'recall': recall_score(
            labels,
            predictions,
            labels=[0, 1],
            average='macro',
            zero_division=0
        ),
    }


def evaluate_on_test(
    trainer,
    test_dataset
):


    output = trainer.predict(
        test_dataset
    )

    predictions = np.argmax(
        output.predictions,
        axis=-1
    )

    labels = output.label_ids

    accuracy = accuracy_score(
        labels,
        predictions
    )

    f1_macro = f1_score(
        labels,
        predictions,
        labels=[0, 1],
        average='macro',
        zero_division=0
    )

    precision_macro = precision_score(
        labels,
        predictions,
        labels=[0, 1],
        average='macro',
        zero_division=0
    )

    recall_macro = recall_score(
        labels,
        predictions,
        labels=[0, 1],
        average='macro',
        zero_division=0
    )

    report = classification_report(
        labels,
        predictions,
        labels=[0, 1],
        target_names=[
            'negative',
            'positive'
        ],
        digits=4,
        zero_division=0
    )

    return (
        accuracy,
        f1_macro,
        precision_macro,
        recall_macro,
        report
    )


def count_trainable_params(model):


    total_params = 0
    trainable_params = 0

    for parameter in model.parameters():

        parameter_count = parameter.numel()

        # Each stored 8-bit value contains two 4-bit parameters
        if parameter.__class__.__name__ == 'Params4bit':
            parameter_count *= 2

        total_params += parameter_count

        if parameter.requires_grad:
            trainable_params += parameter_count

    trainable_percentage = (
        trainable_params
        / total_params
        * 100
    )

    return (
        trainable_params,
        total_params,
        trainable_percentage
    )


def measure_inference_time(
    model,
    tokenizer,
    texts,
    max_length,
    n_warmup=5,
    n_repeat=20
):


    model.eval()

    sample = list(texts)[:32]

    if len(sample) == 0:
        raise ValueError(
            'No Test samples are available for timing.'
        )

    encoded_inputs = tokenizer(
        sample,
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    ).to(model.device)

    with torch.no_grad():
        for _ in range(n_warmup):
            model(**encoded_inputs)

    torch.cuda.synchronize()

    start_time = time.time()

    with torch.no_grad():
        for _ in range(n_repeat):
            model(**encoded_inputs)

    torch.cuda.synchronize()

    elapsed_time = (
        time.time() - start_time
    ) / n_repeat

    inference_ms_per_sample = (
        elapsed_time
        / len(sample)
        * 1000
    )

    return round(
        inference_ms_per_sample,
        3
    )


print('Evaluation utilities ready.')


In [ ]:


from transformers.trainer_utils import get_last_checkpoint


class TrainingTimeCallback(TrainerCallback):
    """
    Measure pure training-step time.

    Validation, logging, data loading, and checkpoint saving
    are excluded.

    The accumulated training time is saved and restored
    when training resumes from a checkpoint.
    """

    def __init__(self):
        self.total_train_time_sec = 0.0
        self._step_start = None
        self._time_state_path = None


    def on_train_begin(
        self,
        args,
        state,
        control,
        **kwargs
    ):
        """
        Restore the previous training time when resuming.
        """

        self._step_start = None

        self._time_state_path = os.path.join(
            args.output_dir,
            'training_time_state.json'
        )

        last_checkpoint = None

        if os.path.isdir(args.output_dir):
            last_checkpoint = get_last_checkpoint(
                args.output_dir
            )

        if (
            last_checkpoint is not None
            and os.path.exists(self._time_state_path)
        ):
            try:
                with open(
                    self._time_state_path,
                    'r',
                    encoding='utf-8'
                ) as file:
                    saved_state = json.load(file)

                self.total_train_time_sec = float(
                    saved_state.get(
                        'total_train_time_sec',
                        0.0
                    )
                )

                print(
                    'Restored cumulative training time: '
                    f'{self.total_train_time_sec:.2f} seconds'
                )

            except Exception as error:
                print(
                    'WARNING: Previous training time '
                    f'could not be restored: {error}'
                )

                self.total_train_time_sec = 0.0

        else:
            self.total_train_time_sec = 0.0


    def on_step_begin(
        self,
        args,
        state,
        control,
        **kwargs
    ):
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        self._step_start = time.perf_counter()


    def on_step_end(
        self,
        args,
        state,
        control,
        **kwargs
    ):
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        if self._step_start is not None:
            self.total_train_time_sec += (
                time.perf_counter()
                - self._step_start
            )

            self._step_start = None


    def _save_time_state(self):
        """
        Save the accumulated training time safely.
        """

        if self._time_state_path is None:
            return

        os.makedirs(
            os.path.dirname(
                self._time_state_path
            ),
            exist_ok=True
        )

        temporary_path = (
            self._time_state_path + '.tmp'
        )

        with open(
            temporary_path,
            'w',
            encoding='utf-8'
        ) as file:
            json.dump(
                {
                    'total_train_time_sec':
                        self.total_train_time_sec
                },
                file,
                indent=2
            )

        os.replace(
            temporary_path,
            self._time_state_path
        )


    def on_save(
        self,
        args,
        state,
        control,
        **kwargs
    ):
        self._save_time_state()


    def on_train_end(
        self,
        args,
        state,
        control,
        **kwargs
    ):
        self._save_time_state()


class EpochMetricsCallback(TrainerCallback):
    """
    Collect validation metrics after every epoch.
    """

    def __init__(
        self,
        method,
        data_level,
        seed
    ):
        self.method = method
        self.data_level = data_level
        self.seed = seed
        self.rows = []


    def on_evaluate(
        self,
        args,
        state,
        control,
        metrics=None,
        **kwargs
    ):
        if metrics is None:
            return

        row = {
            'method': self.method,
            'data_level': self.data_level,
            'seed': self.seed,

            'epoch': (
                float(state.epoch)
                if state.epoch is not None
                else np.nan
            ),

            'step': int(
                state.global_step
            ),

            'eval_loss': metrics.get(
                'eval_loss',
                np.nan
            ),

            'eval_accuracy': metrics.get(
                'eval_accuracy',
                np.nan
            ),

            'eval_f1_macro': metrics.get(
                'eval_f1_macro',
                np.nan
            ),

            'eval_precision_macro': metrics.get(
                'eval_precision',
                np.nan
            ),

            'eval_recall_macro': metrics.get(
                'eval_recall',
                np.nan
            ),
        }

        self.rows.append(row)

        print(
            f'  [Epoch {row["epoch"]:.2f}] '
            f'loss={row["eval_loss"]:.4f} | '
            f'acc={row["eval_accuracy"]:.4f} | '
            f'macro-F1={row["eval_f1_macro"]:.4f} | '
            f'P={row["eval_precision_macro"]:.4f} | '
            f'R={row["eval_recall_macro"]:.4f}'
        )


print('TrainingTimeCallback is ready.')
print('EpochMetricsCallback is ready.')
print(
    'Training time excludes validation, logging, '
    'and checkpoint saving.'
)

In [ ]:

CHECKPOINT_ROOT = os.path.join(
    RECOVERY_ROOT,
    'active_checkpoints'
)

PROGRESS_DIR = os.path.join(
    RECOVERY_ROOT,
    'progress'
)

os.makedirs(
    CHECKPOINT_ROOT,
    exist_ok=True
)

os.makedirs(
    PROGRESS_DIR,
    exist_ok=True
)




RESULTS_PROGRESS_PATH = os.path.join(
    PROGRESS_DIR,
    'completed_results.csv'
)

EPOCH_PROGRESS_PATH = os.path.join(
    PROGRESS_DIR,
    'epoch_metrics.csv'
)

REPORTS_PROGRESS_PATH = os.path.join(
    PROGRESS_DIR,
    'classification_reports.csv'
)




def load_csv_records(file_path):
    """
    Load previously saved records from a CSV file.
    """

    if not os.path.exists(file_path):
        return []

    if os.path.getsize(file_path) == 0:
        return []

    return (
        pd.read_csv(file_path)
        .to_dict('records')
    )


all_results = load_csv_records(
    RESULTS_PROGRESS_PATH
)

all_epoch_metrics = load_csv_records(
    EPOCH_PROGRESS_PATH
)

all_classification_reports = load_csv_records(
    REPORTS_PROGRESS_PATH
)


print('=' * 70)
print('PREVIOUS PROGRESS LOADED')
print(f'Completed runs        : {len(all_results)}')
print(f'Epoch metric records  : {len(all_epoch_metrics)}')
print(
    f'Classification reports: '
    f'{len(all_classification_reports)}'
)
print('=' * 70)


# ============================================================
# Save Progress Safely
# ============================================================

def atomic_save_records(
    records,
    file_path,
    duplicate_columns
):
    """
    Save records safely and remove duplicate records.
    """

    if not records:
        return

    dataframe = pd.DataFrame(
        records
    )

    dataframe = dataframe.drop_duplicates(
        subset=duplicate_columns,
        keep='last'
    ).reset_index(drop=True)

    os.makedirs(
        os.path.dirname(file_path),
        exist_ok=True
    )

    temporary_path = (
        file_path + '.tmp'
    )

    dataframe.to_csv(
        temporary_path,
        index=False
    )

    os.replace(
        temporary_path,
        file_path
    )

    records[:] = dataframe.to_dict(
        'records'
    )


def save_experiment_progress():
    """
    Save completed results, epoch metrics,
    and classification reports to Google Drive.
    """

    atomic_save_records(
        records=all_results,
        file_path=RESULTS_PROGRESS_PATH,
        duplicate_columns=[
            'method',
            'data_level',
            'seed'
        ]
    )

    atomic_save_records(
        records=all_epoch_metrics,
        file_path=EPOCH_PROGRESS_PATH,
        duplicate_columns=[
            'method',
            'data_level',
            'seed',
            'epoch',
            'step'
        ]
    )

    atomic_save_records(
        records=all_classification_reports,
        file_path=REPORTS_PROGRESS_PATH,
        duplicate_columns=[
            'method',
            'data_level',
            'seed'
        ]
    )

    print(
        'Progress saved to Google Drive | '
        f'completed runs: {len(all_results)}'
    )


print('Persistent recovery directories are ready.')
print(f'Recovery root : {RECOVERY_ROOT}')
print(f'Checkpoints   : {CHECKPOINT_ROOT}')
print(f'Progress      : {PROGRESS_DIR}')

In [ ]:


def make_training_args(
    checkpoint_dir: str,
    seed: int,
    is_qlora: bool = False
) -> TrainingArguments:

    training_kwargs = {
        'output_dir': checkpoint_dir,

        'num_train_epochs': NUM_EPOCHS,
        'per_device_train_batch_size': TRAIN_BATCH,
        'per_device_eval_batch_size': EVAL_BATCH,
        'learning_rate': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'warmup_ratio': WARMUP_RATIO,
        'lr_scheduler_type': 'linear',

        'eval_strategy': 'epoch',
        'logging_strategy': 'epoch',
        'report_to': 'none',

        'save_strategy': 'epoch',
        'save_total_limit': 2,

        'load_best_model_at_end':
            LOAD_BEST_MODEL_AT_END,

        'seed': seed,
        'data_seed': seed,

        'fp16': True,

        'remove_unused_columns': True,
    }

    if is_qlora:
        training_kwargs.update({
            'optim': 'paged_adamw_8bit',
            'gradient_checkpointing': False,
            'ddp_find_unused_parameters': False,
        })

    return TrainingArguments(
        **training_kwargs
    )


print('make_training_args() ready.')
print('  Checkpoint frequency : every epoch')
print('  Checkpoints retained : latest two checkpoints')
print('  Final Test model     : final training epoch')
print(
    f'  LR={LEARNING_RATE} | '
    f'batch={TRAIN_BATCH} | '
    f'epochs={NUM_EPOCHS} | '
    f'warmup={WARMUP_RATIO} | '
    f'weight decay={WEIGHT_DECAY} | '
    f'fp16=True'
)

In [ ]:
print('Loading tokenizer for length analysis...')

_tok = AutoTokenizer.from_pretrained(MODEL_NAME)

lengths = train_df['review'].astype(str).apply(
    lambda x: len(
        _tok.encode(
            x,
            truncation=False
        )
    )
)

p50  = int(lengths.quantile(0.50))
p95  = int(lengths.quantile(0.95))
p99  = int(lengths.quantile(0.99))
pmax = int(lengths.max())

print('Token length stats (full training set):')
print(f'  Median (p50) : {p50}')
print(f'  p95          : {p95}')
print(f'  p99          : {p99}')
print(f'  Max          : {pmax}')

MAX_SEQ_LENGTH = min(
    512,
    ((p95 // 32) + 1) * 32
)

print(f'\nMAX_SEQ_LENGTH = {MAX_SEQ_LENGTH}')
print('Fixed for ALL data levels, seeds, and methods.')

del _tok

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f'Tokenizer          : {MODEL_NAME}')
print(f'Vocabulary size    : {tokenizer.vocab_size}')
print(f'Max sequence length: {MAX_SEQ_LENGTH}')


class ArabicSentimentDataset(Dataset):
    """
    Shared across all methods, data levels, and seeds.
    """

    def __init__(
        self,
        df: pd.DataFrame,
        tokenizer,
        max_length: int
    ):
        if max_length is None:
            raise ValueError(
                'MAX_SEQ_LENGTH has not been determined.'
            )

        self.encodings = tokenizer(
            df['review'].astype(str).tolist(),
            max_length=max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )

        self.labels = torch.tensor(
            df['label'].tolist(),
            dtype=torch.long
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings[
                'attention_mask'
            ][idx],
            'labels': self.labels[idx],
        }


print('ArabicSentimentDataset ready.')


In [ ]:
print('Warming up GPU...')

_m = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
).to('cuda')

_d = torch.zeros(
    1,
    MAX_SEQ_LENGTH,
    dtype=torch.long
).to('cuda')

with torch.no_grad():
    _m(
        input_ids=_d,
        attention_mask=_d
    )

del _m, _d

torch.cuda.empty_cache()

print('GPU warm-up complete.')

In [ ]:


def get_training_data(data_level, seed):
    """
    Select the correct training data.
    """

    if data_level == 'full_dataset':
        return train_df.copy()

    if data_level in [
        '25_per_class',
        '100_per_class'
    ]:
        return (
            REDUCED_DATA_BY_SEED[seed][data_level]
            .copy()
        )

    raise ValueError(
        f'Unknown data level: {data_level}'
    )


def collect_epoch_metrics_from_trainer(
    trainer,
    method,
    data_level,
    seed
):
    """
    Collect validation metrics from Trainer history.
    """

    rows = []

    for record in trainer.state.log_history:

        if 'eval_loss' not in record:
            continue

        rows.append({
            'method': method,
            'data_level': data_level,
            'seed': seed,
            'epoch': record.get('epoch', np.nan),
            'step': record.get('step', np.nan),
            'eval_loss':
                record.get('eval_loss', np.nan),
            'eval_accuracy':
                record.get('eval_accuracy', np.nan),
            'eval_f1_macro':
                record.get('eval_f1_macro', np.nan),
            'eval_precision_macro':
                record.get('eval_precision', np.nan),
            'eval_recall_macro':
                record.get('eval_recall', np.nan),
        })

    return rows


def run_full_finetuning(
    data_level,
    seed,
    results_store,
    epoch_metrics_store,
    reports_store
):
    """
    Run Full Fine-Tuning for one data level and seed.
    """

    method = 'full_finetuning'

    trainer = None
    model = None

    checkpoint_dir = os.path.join(
        CHECKPOINT_ROOT,
        method,
        data_level,
        f'seed_{seed}'
    )

    os.makedirs(
        checkpoint_dir,
        exist_ok=True
    )

    try:
        set_seed(seed)

        train_data = get_training_data(
            data_level=data_level,
            seed=seed
        )

        print('\n' + '=' * 70)
        print('FULL FINE-TUNING')
        print(f'Data level : {data_level}')
        print(f'Seed       : {seed}')
        print(f'N train    : {len(train_data)}')
        print('=' * 70)

        train_dataset = ArabicSentimentDataset(
            train_data,
            tokenizer,
            MAX_SEQ_LENGTH
        )

        validation_dataset = ArabicSentimentDataset(
            val_df,
            tokenizer,
            MAX_SEQ_LENGTH
        )

        test_dataset = ArabicSentimentDataset(
            test_df,
            tokenizer,
            MAX_SEQ_LENGTH
        )

        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

        print('Loading CAMeLBERT model...')

        model = (
            AutoModelForSequenceClassification
            .from_pretrained(
                MODEL_NAME,
                num_labels=NUM_LABELS
            )
            .to('cuda')
        )

        vram_at_load_mb = (
            torch.cuda.memory_allocated()
            / (1024 ** 2)
        )

        (
            trainable_params,
            total_params,
            trainable_percentage
        ) = count_trainable_params(model)

        print(
            f'Trainable parameters: '
            f'{trainable_params:,} / '
            f'{total_params:,} '
            f'({trainable_percentage:.4f}%)'
        )

        print(
            f'VRAM after model loading: '
            f'{vram_at_load_mb:.2f} MB'
        )

        time_callback = TrainingTimeCallback()

        epoch_callback = EpochMetricsCallback(
            method=method,
            data_level=data_level,
            seed=seed
        )

        trainer = Trainer(
            model=model,

            args=make_training_args(
                checkpoint_dir=checkpoint_dir,
                seed=seed,
                is_qlora=False
            ),

            train_dataset=train_dataset,
            eval_dataset=validation_dataset,

            compute_metrics=compute_metrics,

            callbacks=[
                time_callback,
                epoch_callback
            ]
        )

        last_checkpoint = get_last_checkpoint(
            checkpoint_dir
        )

        if last_checkpoint is not None:
            print(
                'Checkpoint found. Resuming from:\n'
                f'{last_checkpoint}'
            )

            trainer.train(
                resume_from_checkpoint=last_checkpoint
            )

        else:
            print(
                'No checkpoint found. '
                'Starting from the beginning.'
            )

            trainer.train()

        training_time_sec = (
            time_callback.total_train_time_sec
        )

        peak_vram_mb = (
            torch.cuda.max_memory_allocated()
            / (1024 ** 2)
        )

        epoch_metrics_store.extend(
            collect_epoch_metrics_from_trainer(
                trainer=trainer,
                method=method,
                data_level=data_level,
                seed=seed
            )
        )

        (
            test_accuracy,
            test_f1_macro,
            test_precision_macro,
            test_recall_macro,
            report
        ) = evaluate_on_test(
            trainer,
            test_dataset
        )

        inference_ms = measure_inference_time(
            model=trainer.model,
            tokenizer=tokenizer,
            texts=test_df['review'].tolist(),
            max_length=MAX_SEQ_LENGTH
        )

        print('\nRun summary')
        print('-' * 70)
        print(f'Test Accuracy  : {test_accuracy:.4f}')
        print(f'Test Macro-F1  : {test_f1_macro:.4f}')
        print(f'Test Precision : {test_precision_macro:.4f}')
        print(f'Test Recall    : {test_recall_macro:.4f}')
        print(f'Training time  : {training_time_sec:.2f} seconds')
        print(f'Peak VRAM      : {peak_vram_mb:.2f} MB')
        print(f'Inference      : {inference_ms:.3f} ms/sample')
        print('Test model     : final training epoch')

        print(
            f'\nClassification report:\n{report}'
        )

        result = {
            'method': method,
            'data_level': data_level,
            'seed': seed,
            'n_train_samples': len(train_data),

            'test_accuracy': test_accuracy,
            'test_f1_macro': test_f1_macro,
            'test_precision_macro':
                test_precision_macro,
            'test_recall_macro':
                test_recall_macro,

            'training_time_sec':
                training_time_sec,
            'peak_vram_mb':
                peak_vram_mb,
            'vram_at_load_mb':
                vram_at_load_mb,

            'trainable_params':
                trainable_params,
            'total_params':
                total_params,
            'trainable_percentage':
                trainable_percentage,

            'inference_ms':
                inference_ms,
        }

        results_store.append(
            result
        )

        reports_store.append({
            'method': method,
            'data_level': data_level,
            'seed': seed,
            'classification_report': report,
        })

        save_experiment_progress()

        print(
            'Full Fine-Tuning result saved successfully.'
        )

        return result

    except Exception as error:
        print(
            '\nFull Fine-Tuning failed | '
            f'data={data_level} | seed={seed}'
        )

        print(f'Error: {error}')

        raise

    finally:
        if trainer is not None:
            del trainer

        if model is not None:
            del model

        if 'train_dataset' in locals():
            del train_dataset

        if 'validation_dataset' in locals():
            del validation_dataset

        if 'test_dataset' in locals():
            del test_dataset

        gc.collect()
        torch.cuda.empty_cache()


print('Full Fine-Tuning function is ready.')

In [ ]:


import shutil
import gc


FULL_FT_METHOD = 'full_finetuning'

DATA_LEVEL_ORDER = [
    'full_dataset',
    '25_per_class',
    '100_per_class',
]


def run_completed(
    results_store,
    method,
    data_level,
    seed
):


    return any(
        result.get('method') == method
        and result.get('data_level') == data_level
        and int(result.get('seed')) == int(seed)
        for result in results_store
    )


def delete_completed_checkpoint(
    method,
    data_level,
    seed
):
    """
    Delete the checkpoint only after the result
    has been saved successfully.
    """

    checkpoint_dir = os.path.abspath(
        os.path.join(
            CHECKPOINT_ROOT,
            method,
            data_level,
            f'seed_{seed}'
        )
    )

    checkpoint_root = os.path.abspath(
        CHECKPOINT_ROOT
    )

    if os.path.commonpath([
        checkpoint_root,
        checkpoint_dir
    ]) != checkpoint_root:
        raise ValueError(
            'Unsafe checkpoint path detected.'
        )

    if os.path.isdir(checkpoint_dir):
        shutil.rmtree(
            checkpoint_dir
        )

        print(
            'Completed checkpoint deleted: '
            f'{method} | {data_level} | seed={seed}'
        )


def save_full_finetuning_results():
    """
    Save Full Fine-Tuning results in a separate CSV.
    """

    method_results = [
        result
        for result in all_results
        if result.get('method') == FULL_FT_METHOD
    ]

    method_results_df = pd.DataFrame(
        method_results
    )

    method_results_df = (
        method_results_df
        .drop_duplicates(
            subset=[
                'method',
                'data_level',
                'seed'
            ],
            keep='last'
        )
        .reset_index(drop=True)
    )

    method_results_dir = os.path.join(
        RECOVERY_ROOT,
        'method_results'
    )

    os.makedirs(
        method_results_dir,
        exist_ok=True
    )

    results_path = os.path.join(
        method_results_dir,
        'results_full_finetuning.csv'
    )

    temporary_path = (
        results_path + '.tmp'
    )

    method_results_df.to_csv(
        temporary_path,
        index=False
    )

    os.replace(
        temporary_path,
        results_path
    )

    return (
        method_results_df,
        results_path
    )


TOTAL_FULL_FT_RUNS = (
    len(DATA_LEVEL_ORDER)
    * len(SEEDS)
)

completed_count = 0


print('=' * 70)
print('FULL FINE-TUNING EXPERIMENT')
print(f'Data levels : {len(DATA_LEVEL_ORDER)}')
print(f'Seeds       : {len(SEEDS)}')
print(f'Total runs  : {TOTAL_FULL_FT_RUNS}')
print('=' * 70)


for data_level in DATA_LEVEL_ORDER:

    for seed in SEEDS:

        if run_completed(
            results_store=all_results,
            method=FULL_FT_METHOD,
            data_level=data_level,
            seed=seed
        ):
            completed_count += 1

            print(
                '\nSkipping completed run: '
                f'{FULL_FT_METHOD} | '
                f'{data_level} | seed={seed}'
            )


            delete_completed_checkpoint(
                method=FULL_FT_METHOD,
                data_level=data_level,
                seed=seed
            )

            continue

        print(
            f'\nStarting Full Fine-Tuning run '
            f'{completed_count + 1}/'
            f'{TOTAL_FULL_FT_RUNS}: '
            f'{data_level} | seed={seed}'
        )

        run_full_finetuning(
            data_level=data_level,
            seed=seed,
            results_store=all_results,
            epoch_metrics_store=all_epoch_metrics,
            reports_store=all_classification_reports
        )


        delete_completed_checkpoint(
            method=FULL_FT_METHOD,
            data_level=data_level,
            seed=seed
        )

        completed_count += 1

        full_ft_results_df, full_ft_results_path = (
            save_full_finetuning_results()
        )

        print(
            f'Full Fine-Tuning progress: '
            f'{completed_count}/'
            f'{TOTAL_FULL_FT_RUNS}'
        )

        gc.collect()
        torch.cuda.empty_cache()


full_ft_results_df, full_ft_results_path = (
    save_full_finetuning_results()
)


if len(full_ft_results_df) != TOTAL_FULL_FT_RUNS:
    print(
        '\nFull Fine-Tuning is not complete yet: '
        f'{len(full_ft_results_df)}/'
        f'{TOTAL_FULL_FT_RUNS} runs.'
    )

else:
    print('\n' + '=' * 70)
    print('ALL FULL FINE-TUNING RUNS COMPLETED')
    print(
        f'Completed runs: '
        f'{len(full_ft_results_df)}/'
        f'{TOTAL_FULL_FT_RUNS}'
    )
    print(
        f'Results saved to:\n'
        f'{full_ft_results_path}'
    )
    print('=' * 70)

    display(
        full_ft_results_df[
            [
                'data_level',
                'seed',
                'test_accuracy',
                'test_f1_macro',
                'test_precision_macro',
                'test_recall_macro',
                'training_time_sec',
                'peak_vram_mb',
                'inference_ms',
            ]
        ]
    )


In [ ]:


def calculate_mean_sd_ci(
    values,
    confidence=0.95
):
    """
    Calculate the mean, sample SD, and
    Student-t confidence interval.
    """

    values = (
        pd.Series(values)
        .dropna()
        .astype(float)
    )

    n = len(values)

    if n == 0:
        return {
            'n': 0,
            'mean': np.nan,
            'sd': np.nan,
            'ci_lower': np.nan,
            'ci_upper': np.nan,
        }

    mean = values.mean()

    if n == 1:
        return {
            'n': 1,
            'mean': mean,
            'sd': np.nan,
            'ci_lower': np.nan,
            'ci_upper': np.nan,
        }

    # Sample standard deviation: denominator n - 1
    sample_sd = values.std(
        ddof=1
    )

    standard_error = (
        sample_sd
        / np.sqrt(n)
    )

    degrees_of_freedom = (
        n - 1
    )

    probability = (
        1 + confidence
    ) / 2

    t_critical = t.ppf(
        probability,
        degrees_of_freedom
    )

    margin_of_error = (
        t_critical
        * standard_error
    )

    return {
        'n': n,
        'mean': mean,
        'sd': sample_sd,
        'ci_lower':
            mean - margin_of_error,
        'ci_upper':
            mean + margin_of_error,
    }


def format_mean_sd_ci(
    values,
    decimals=4
):
    """
    Format:
    Mean ± Sample SD [95% CI lower, upper]
    """

    statistics = calculate_mean_sd_ci(
        values=values,
        confidence=0.95
    )

    if statistics['n'] < 2:
        return 'Insufficient runs'

    return (
        f'{statistics["mean"]:.{decimals}f} '
        f'± {statistics["sd"]:.{decimals}f} '
        f'['
        f'{statistics["ci_lower"]:.{decimals}f}, '
        f'{statistics["ci_upper"]:.{decimals}f}'
        f']'
    )



FULL_FT_RESULTS_PATH = os.path.join(
    RECOVERY_ROOT,
    'method_results',
    'results_full_finetuning.csv'
)

if not os.path.exists(
    FULL_FT_RESULTS_PATH
):
    raise FileNotFoundError(
        'Full Fine-Tuning results were not found.'
    )

full_ft_results = pd.read_csv(
    FULL_FT_RESULTS_PATH
)


full_ft_results = (
    full_ft_results
    .drop_duplicates(
        subset=[
            'method',
            'data_level',
            'seed'
        ],
        keep='last'
    )
    .reset_index(drop=True)
)



expected_runs = (
    len(DATA_LEVELS)
    * len(SEEDS)
)

if len(full_ft_results) != expected_runs:
    raise ValueError(
        f'Expected {expected_runs} Full Fine-Tuning runs, '
        f'but found {len(full_ft_results)}.'
    )


for data_level in DATA_LEVELS:

    completed_seeds = set(
        full_ft_results.loc[
            full_ft_results['data_level']
            == data_level,
            'seed'
        ].astype(int)
    )

    if completed_seeds != set(SEEDS):
        raise ValueError(
            f'Incorrect Seeds for {data_level}. '
            f'Found: {sorted(completed_seeds)}'
        )




summary_rows = []

for data_level in [
    'full_dataset',
    '25_per_class',
    '100_per_class',
]:

    condition = full_ft_results[
        full_ft_results['data_level']
        == data_level
    ]

    summary_rows.append({
        'Method':
            'Full Fine-Tuning',

        'Data Level':
            data_level,

        'N Train':
            int(
                condition[
                    'n_train_samples'
                ].iloc[0]
            ),

        'Seeds':
            len(condition),

        'Accuracy':
            format_mean_sd_ci(
                condition['test_accuracy'],
                decimals=4
            ),

        'Macro-F1':
            format_mean_sd_ci(
                condition['test_f1_macro'],
                decimals=4
            ),

        'Precision':
            format_mean_sd_ci(
                condition[
                    'test_precision_macro'
                ],
                decimals=4
            ),

        'Recall':
            format_mean_sd_ci(
                condition[
                    'test_recall_macro'
                ],
                decimals=4
            ),

        'Training Time (s)':
            format_mean_sd_ci(
                condition['training_time_sec'],
                decimals=2
            ),

        'Peak VRAM (MB)':
            format_mean_sd_ci(
                condition['peak_vram_mb'],
                decimals=2
            ),

        'Inference (ms/sample)':
            format_mean_sd_ci(
                condition['inference_ms'],
                decimals=3
            ),

        'Trainable Parameters':
            int(
                condition[
                    'trainable_params'
                ].iloc[0]
            ),

        'Total Parameters':
            int(
                condition[
                    'total_params'
                ].iloc[0]
            ),

        'Trainable %':
            float(
                condition[
                    'trainable_percentage'
                ].iloc[0]
            ),
    })


full_ft_summary = pd.DataFrame(
    summary_rows
)



print('\n' + '=' * 140)
print('FULL FINE-TUNING — STATISTICAL SUMMARY')
print(
    'Mean ± Sample SD '
    '[Student-t 95% Confidence Interval]'
)
print(f'Seeds per data level: {len(SEEDS)}')
print('=' * 140)

display(
    full_ft_summary
)


FULL_FT_SUMMARY_PATH = os.path.join(
    RECOVERY_ROOT,
    'method_results',
    'summary_full_finetuning.csv'
)

full_ft_summary.to_csv(
    FULL_FT_SUMMARY_PATH,
    index=False
)

print(
    'Full Fine-Tuning summary saved to:\n'
    f'{FULL_FT_SUMMARY_PATH}'
)

In [ ]:
# ============================================================
# Frozen Backbone
# ============================================================

def run_frozen_backbone(
    data_level,
    seed,
    results_store,
    epoch_metrics_store,
    reports_store
):
    """
    Train only the classification head while keeping
    the pretrained CAMeLBERT backbone frozen.
    """

    method = 'frozen_backbone'

    trainer = None
    model = None

    checkpoint_dir = os.path.join(
        CHECKPOINT_ROOT,
        method,
        data_level,
        f'seed_{seed}'
    )

    os.makedirs(
        checkpoint_dir,
        exist_ok=True
    )

    try:
        set_seed(seed)

        train_data = get_training_data(
            data_level=data_level,
            seed=seed
        )

        print('\n' + '=' * 70)
        print('FROZEN BACKBONE')
        print(f'Data level : {data_level}')
        print(f'Seed       : {seed}')
        print(f'N train    : {len(train_data)}')
        print('=' * 70)

        train_dataset = ArabicSentimentDataset(
            train_data,
            tokenizer,
            MAX_SEQ_LENGTH
        )

        validation_dataset = ArabicSentimentDataset(
            val_df,
            tokenizer,
            MAX_SEQ_LENGTH
        )

        test_dataset = ArabicSentimentDataset(
            test_df,
            tokenizer,
            MAX_SEQ_LENGTH
        )

        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

        print('Loading CAMeLBERT model...')

        model = (
            AutoModelForSequenceClassification
            .from_pretrained(
                MODEL_NAME,
                num_labels=NUM_LABELS
            )
            .to('cuda')
        )

        # Freeze the pretrained CAMeLBERT backbone
        for parameter in model.base_model.parameters():
            parameter.requires_grad = False

        print(
            'CAMeLBERT backbone frozen; '
            'only the classification head is trainable.'
        )

        vram_at_load_mb = (
            torch.cuda.memory_allocated()
            / (1024 ** 2)
        )

        (
            trainable_params,
            total_params,
            trainable_percentage
        ) = count_trainable_params(
            model
        )

        print(
            f'Trainable parameters: '
            f'{trainable_params:,} / '
            f'{total_params:,} '
            f'({trainable_percentage:.4f}%)'
        )

        print(
            f'VRAM after model loading: '
            f'{vram_at_load_mb:.2f} MB'
        )

        time_callback = (
            TrainingTimeCallback()
        )

        epoch_callback = EpochMetricsCallback(
            method=method,
            data_level=data_level,
            seed=seed
        )

        trainer = Trainer(
            model=model,

            args=make_training_args(
                checkpoint_dir=checkpoint_dir,
                seed=seed,
                is_qlora=False
            ),

            train_dataset=train_dataset,
            eval_dataset=validation_dataset,

            compute_metrics=compute_metrics,

            callbacks=[
                time_callback,
                epoch_callback
            ]
        )

        last_checkpoint = get_last_checkpoint(
            checkpoint_dir
        )

        if last_checkpoint is not None:
            print(
                'Checkpoint found. Resuming from:\n'
                f'{last_checkpoint}'
            )

            trainer.train(
                resume_from_checkpoint=last_checkpoint
            )

        else:
            print(
                'No checkpoint found. '
                'Starting from the beginning.'
            )

            trainer.train()

        training_time_sec = (
            time_callback.total_train_time_sec
        )

        peak_vram_mb = (
            torch.cuda.max_memory_allocated()
            / (1024 ** 2)
        )

        epoch_metrics_store.extend(
            collect_epoch_metrics_from_trainer(
                trainer=trainer,
                method=method,
                data_level=data_level,
                seed=seed
            )
        )

        (
            test_accuracy,
            test_f1_macro,
            test_precision_macro,
            test_recall_macro,
            report
        ) = evaluate_on_test(
            trainer,
            test_dataset
        )

        inference_ms = measure_inference_time(
            model=trainer.model,
            tokenizer=tokenizer,
            texts=test_df['review'].tolist(),
            max_length=MAX_SEQ_LENGTH
        )

        print('\nRun summary')
        print('-' * 70)
        print(
            f'Test Accuracy  : '
            f'{test_accuracy:.4f}'
        )
        print(
            f'Test Macro-F1  : '
            f'{test_f1_macro:.4f}'
        )
        print(
            f'Test Precision : '
            f'{test_precision_macro:.4f}'
        )
        print(
            f'Test Recall    : '
            f'{test_recall_macro:.4f}'
        )
        print(
            f'Training time  : '
            f'{training_time_sec:.2f} seconds'
        )
        print(
            f'Peak VRAM      : '
            f'{peak_vram_mb:.2f} MB'
        )
        print(
            f'Inference      : '
            f'{inference_ms:.3f} ms/sample'
        )
        print(
            'Test model     : '
            'final training epoch'
        )

        print(
            f'\nClassification report:\n'
            f'{report}'
        )

        result = {
            'method': method,
            'data_level': data_level,
            'seed': seed,
            'n_train_samples': len(train_data),

            'test_accuracy':
                test_accuracy,
            'test_f1_macro':
                test_f1_macro,
            'test_precision_macro':
                test_precision_macro,
            'test_recall_macro':
                test_recall_macro,

            'training_time_sec':
                training_time_sec,
            'peak_vram_mb':
                peak_vram_mb,
            'vram_at_load_mb':
                vram_at_load_mb,

            'trainable_params':
                trainable_params,
            'total_params':
                total_params,
            'trainable_percentage':
                trainable_percentage,

            'inference_ms':
                inference_ms,
        }

        results_store.append(
            result
        )

        reports_store.append({
            'method': method,
            'data_level': data_level,
            'seed': seed,
            'classification_report':
                report,
        })

        # Save immediately after the run succeeds
        save_experiment_progress()

        print(
            'Frozen-Backbone result '
            'saved successfully.'
        )

        return result

    except Exception as error:
        print(
            '\nFrozen-Backbone run failed | '
            f'data={data_level} | seed={seed}'
        )

        print(f'Error: {error}')

        # Keep the checkpoint for recovery
        raise

    finally:
        if trainer is not None:
            del trainer

        if model is not None:
            del model

        if 'train_dataset' in locals():
            del train_dataset

        if 'validation_dataset' in locals():
            del validation_dataset

        if 'test_dataset' in locals():
            del test_dataset

        gc.collect()
        torch.cuda.empty_cache()


print('Frozen Backbone function is ready.')

In [ ]:

import shutil
import gc


FROZEN_METHOD = 'frozen_backbone'

FROZEN_DATA_LEVEL_ORDER = [
    'full_dataset',
    '25_per_class',
    '100_per_class',
]


def frozen_run_completed(
    data_level,
    seed
):


    return any(
        result.get('method') == FROZEN_METHOD
        and result.get('data_level') == data_level
        and int(result.get('seed')) == int(seed)
        for result in all_results
    )


def delete_frozen_checkpoint(
    data_level,
    seed
):


    checkpoint_dir = os.path.abspath(
        os.path.join(
            CHECKPOINT_ROOT,
            FROZEN_METHOD,
            data_level,
            f'seed_{seed}'
        )
    )

    checkpoint_root = os.path.abspath(
        CHECKPOINT_ROOT
    )

    if os.path.commonpath([
        checkpoint_root,
        checkpoint_dir
    ]) != checkpoint_root:
        raise ValueError(
            'Unsafe checkpoint path detected.'
        )

    if os.path.isdir(checkpoint_dir):
        shutil.rmtree(
            checkpoint_dir
        )

        print(
            'Completed checkpoint deleted: '
            f'{FROZEN_METHOD} | '
            f'{data_level} | seed={seed}'
        )


def save_frozen_results():
    """
    Save Frozen-Backbone results in a separate CSV.
    """

    frozen_records = [
        result
        for result in all_results
        if result.get('method') == FROZEN_METHOD
    ]

    frozen_results_df = pd.DataFrame(
        frozen_records
    )

    frozen_results_df = (
        frozen_results_df
        .drop_duplicates(
            subset=[
                'method',
                'data_level',
                'seed'
            ],
            keep='last'
        )
        .reset_index(drop=True)
    )

    method_results_dir = os.path.join(
        RECOVERY_ROOT,
        'method_results'
    )

    os.makedirs(
        method_results_dir,
        exist_ok=True
    )

    results_path = os.path.join(
        method_results_dir,
        'results_frozen_backbone.csv'
    )

    temporary_path = (
        results_path + '.tmp'
    )

    frozen_results_df.to_csv(
        temporary_path,
        index=False
    )

    os.replace(
        temporary_path,
        results_path
    )

    return (
        frozen_results_df,
        results_path
    )


TOTAL_FROZEN_RUNS = (
    len(FROZEN_DATA_LEVEL_ORDER)
    * len(SEEDS)
)

completed_count = 0


print('=' * 70)
print('FROZEN-BACKBONE EXPERIMENT')
print(
    f'Data levels : '
    f'{len(FROZEN_DATA_LEVEL_ORDER)}'
)
print(f'Seeds       : {len(SEEDS)}')
print(f'Total runs  : {TOTAL_FROZEN_RUNS}')
print('=' * 70)


for data_level in FROZEN_DATA_LEVEL_ORDER:

    for seed in SEEDS:

        if frozen_run_completed(
            data_level=data_level,
            seed=seed
        ):
            completed_count += 1

            print(
                '\nSkipping completed run: '
                f'{FROZEN_METHOD} | '
                f'{data_level} | seed={seed}'
            )

            delete_frozen_checkpoint(
                data_level=data_level,
                seed=seed
            )

            continue

        print(
            f'\nStarting Frozen run '
            f'{completed_count + 1}/'
            f'{TOTAL_FROZEN_RUNS}: '
            f'{data_level} | seed={seed}'
        )

        run_frozen_backbone(
            data_level=data_level,
            seed=seed,
            results_store=all_results,
            epoch_metrics_store=all_epoch_metrics,
            reports_store=all_classification_reports
        )

        delete_frozen_checkpoint(
            data_level=data_level,
            seed=seed
        )

        completed_count += 1

        frozen_results_df, frozen_results_path = (
            save_frozen_results()
        )

        print(
            f'Frozen progress: '
            f'{completed_count}/'
            f'{TOTAL_FROZEN_RUNS}'
        )

        gc.collect()
        torch.cuda.empty_cache()


frozen_results_df, frozen_results_path = (
    save_frozen_results()
)


if len(frozen_results_df) != TOTAL_FROZEN_RUNS:
    print(
        '\nFrozen Backbone is not complete yet: '
        f'{len(frozen_results_df)}/'
        f'{TOTAL_FROZEN_RUNS} runs.'
    )

else:
    print('\n' + '=' * 70)
    print('ALL FROZEN-BACKBONE RUNS COMPLETED')
    print(
        f'Completed runs: '
        f'{len(frozen_results_df)}/'
        f'{TOTAL_FROZEN_RUNS}'
    )
    print(
        f'Results saved to:\n'
        f'{frozen_results_path}'
    )
    print('=' * 70)

    display(
        frozen_results_df[
            [
                'data_level',
                'seed',
                'test_accuracy',
                'test_f1_macro',
                'test_precision_macro',
                'test_recall_macro',
                'training_time_sec',
                'peak_vram_mb',
                'inference_ms',
            ]
        ]
    )

In [ ]:
FROZEN_RESULTS_PATH = os.path.join(
    RECOVERY_ROOT,
    'method_results',
    'results_frozen_backbone.csv'
)

if not os.path.exists(
    FROZEN_RESULTS_PATH
):
    raise FileNotFoundError(
        'Frozen-Backbone results were not found.'
    )


frozen_results = pd.read_csv(
    FROZEN_RESULTS_PATH
)


frozen_results = (
    frozen_results
    .drop_duplicates(
        subset=[
            'method',
            'data_level',
            'seed'
        ],
        keep='last'
    )
    .reset_index(drop=True)
)


expected_frozen_runs = (
    len(DATA_LEVELS)
    * len(SEEDS)
)

if len(frozen_results) != expected_frozen_runs:
    raise ValueError(
        f'Expected {expected_frozen_runs} '
        f'Frozen-Backbone runs, '
        f'but found {len(frozen_results)}.'
    )


for data_level in DATA_LEVELS:

    completed_seeds = set(
        frozen_results.loc[
            frozen_results['data_level']
            == data_level,
            'seed'
        ].astype(int)
    )

    if completed_seeds != set(SEEDS):
        raise ValueError(
            f'Incorrect Seeds for {data_level}. '
            f'Found: {sorted(completed_seeds)}'
        )




frozen_per_seed_results = (
    frozen_results[
        [
            'data_level',
            'seed',
            'test_accuracy',
            'test_f1_macro',
            'test_precision_macro',
            'test_recall_macro',
            'training_time_sec',
            'peak_vram_mb',
            'inference_ms',
        ]
    ]
    .sort_values(
        by=[
            'data_level',
            'seed'
        ]
    )
    .reset_index(drop=True)
)

print('\nFROZEN BACKBONE — INDIVIDUAL SEED RESULTS')

display(
    frozen_per_seed_results
)



frozen_summary_rows = []

for data_level in [
    'full_dataset',
    '25_per_class',
    '100_per_class',
]:

    condition = frozen_results[
        frozen_results['data_level']
        == data_level
    ]

    frozen_summary_rows.append({
        'Method':
            'Frozen Backbone',

        'Data Level':
            data_level,

        'N Train':
            int(
                condition[
                    'n_train_samples'
                ].iloc[0]
            ),

        'Seeds':
            len(condition),

        'Accuracy':
            format_mean_sd_ci(
                condition['test_accuracy'],
                decimals=4
            ),

        'Macro-F1':
            format_mean_sd_ci(
                condition['test_f1_macro'],
                decimals=4
            ),

        'Precision':
            format_mean_sd_ci(
                condition[
                    'test_precision_macro'
                ],
                decimals=4
            ),

        'Recall':
            format_mean_sd_ci(
                condition[
                    'test_recall_macro'
                ],
                decimals=4
            ),

        'Training Time (s)':
            format_mean_sd_ci(
                condition[
                    'training_time_sec'
                ],
                decimals=2
            ),

        'Peak VRAM (MB)':
            format_mean_sd_ci(
                condition[
                    'peak_vram_mb'
                ],
                decimals=2
            ),

        'Inference (ms/sample)':
            format_mean_sd_ci(
                condition[
                    'inference_ms'
                ],
                decimals=3
            ),

        'Trainable Parameters':
            int(
                condition[
                    'trainable_params'
                ].iloc[0]
            ),

        'Total Parameters':
            int(
                condition[
                    'total_params'
                ].iloc[0]
            ),

        'Trainable %':
            float(
                condition[
                    'trainable_percentage'
                ].iloc[0]
            ),
    })


frozen_summary = pd.DataFrame(
    frozen_summary_rows
)


print('\n' + '=' * 140)
print('FROZEN BACKBONE — STATISTICAL SUMMARY')
print(
    'Mean ± Sample SD '
    '[Student-t 95% Confidence Interval]'
)
print(f'Seeds per data level: {len(SEEDS)}')
print('=' * 140)

display(
    frozen_summary
)


FROZEN_SUMMARY_PATH = os.path.join(
    RECOVERY_ROOT,
    'method_results',
    'summary_frozen_backbone.csv'
)

frozen_summary.to_csv(
    FROZEN_SUMMARY_PATH,
    index=False
)

print(
    'Frozen-Backbone summary saved to:\n'
    f'{FROZEN_SUMMARY_PATH}'
)

In [ ]:
=

def run_lora(
    data_level,
    seed,
    results_store,
    epoch_metrics_store,
    reports_store
):


    method = 'lora'

    trainer = None
    model = None
    base_model = None

    checkpoint_dir = os.path.join(
        CHECKPOINT_ROOT,
        method,
        data_level,
        f'seed_{seed}'
    )

    os.makedirs(
        checkpoint_dir,
        exist_ok=True
    )

    try:
        set_seed(seed)

        train_data = get_training_data(
            data_level=data_level,
            seed=seed
        )

        print('\n' + '=' * 70)
        print('LoRA')
        print(f'Data level : {data_level}')
        print(f'Seed       : {seed}')
        print(f'N train    : {len(train_data)}')
        print('=' * 70)

        train_dataset = ArabicSentimentDataset(
            train_data,
            tokenizer,
            MAX_SEQ_LENGTH
        )

        validation_dataset = ArabicSentimentDataset(
            val_df,
            tokenizer,
            MAX_SEQ_LENGTH
        )

        test_dataset = ArabicSentimentDataset(
            test_df,
            tokenizer,
            MAX_SEQ_LENGTH
        )

        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

        print('Loading CAMeLBERT base model...')

        base_model = (
            AutoModelForSequenceClassification
            .from_pretrained(
                MODEL_NAME,
                num_labels=NUM_LABELS
            )
            .to('cuda')
        )

        print('Applying LoRA adapters...')

        lora_config = LoraConfig(
            task_type=TaskType.SEQ_CLS,

            r=LORA_R,
            lora_alpha=LORA_ALPHA,
            lora_dropout=LORA_DROPOUT,

            target_modules=
                LORA_TARGET_MODULES,

            modules_to_save=[
                'classifier'
            ],

            bias='none',
            inference_mode=False
        )

        model = get_peft_model(
            base_model,
            lora_config
        )

        model.print_trainable_parameters()

        vram_at_load_mb = (
            torch.cuda.memory_allocated()
            / (1024 ** 2)
        )

        (
            trainable_params,
            total_params,
            trainable_percentage
        ) = count_trainable_params(
            model
        )

        print(
            f'Trainable parameters: '
            f'{trainable_params:,} / '
            f'{total_params:,} '
            f'({trainable_percentage:.4f}%)'
        )

        print(
            f'VRAM after model loading: '
            f'{vram_at_load_mb:.2f} MB'
        )

        time_callback = (
            TrainingTimeCallback()
        )

        epoch_callback = EpochMetricsCallback(
            method=method,
            data_level=data_level,
            seed=seed
        )

        trainer = Trainer(
            model=model,

            args=make_training_args(
                checkpoint_dir=checkpoint_dir,
                seed=seed,
                is_qlora=False
            ),

            train_dataset=train_dataset,
            eval_dataset=validation_dataset,

            compute_metrics=compute_metrics,

            callbacks=[
                time_callback,
                epoch_callback
            ]
        )

        last_checkpoint = get_last_checkpoint(
            checkpoint_dir
        )

        if last_checkpoint is not None:
            print(
                'Checkpoint found. Resuming from:\n'
                f'{last_checkpoint}'
            )

            trainer.train(
                resume_from_checkpoint=
                    last_checkpoint
            )

        else:
            print(
                'No checkpoint found. '
                'Starting from the beginning.'
            )

            trainer.train()

        training_time_sec = (
            time_callback.total_train_time_sec
        )

        peak_vram_mb = (
            torch.cuda.max_memory_allocated()
            / (1024 ** 2)
        )

        epoch_metrics_store.extend(
            collect_epoch_metrics_from_trainer(
                trainer=trainer,
                method=method,
                data_level=data_level,
                seed=seed
            )
        )

        (
            test_accuracy,
            test_f1_macro,
            test_precision_macro,
            test_recall_macro,
            report
        ) = evaluate_on_test(
            trainer,
            test_dataset
        )

        inference_ms = measure_inference_time(
            model=trainer.model,
            tokenizer=tokenizer,
            texts=test_df['review'].tolist(),
            max_length=MAX_SEQ_LENGTH
        )

        print('\nRun summary')
        print('-' * 70)
        print(
            f'Test Accuracy  : '
            f'{test_accuracy:.4f}'
        )
        print(
            f'Test Macro-F1  : '
            f'{test_f1_macro:.4f}'
        )
        print(
            f'Test Precision : '
            f'{test_precision_macro:.4f}'
        )
        print(
            f'Test Recall    : '
            f'{test_recall_macro:.4f}'
        )
        print(
            f'Training time  : '
            f'{training_time_sec:.2f} seconds'
        )
        print(
            f'Peak VRAM      : '
            f'{peak_vram_mb:.2f} MB'
        )
        print(
            f'Inference      : '
            f'{inference_ms:.3f} ms/sample'
        )
        print(
            'Test model     : '
            'final training epoch'
        )

        print(
            f'\nClassification report:\n'
            f'{report}'
        )

        result = {
            'method': method,
            'data_level': data_level,
            'seed': seed,
            'n_train_samples': len(train_data),

            'test_accuracy':
                test_accuracy,
            'test_f1_macro':
                test_f1_macro,
            'test_precision_macro':
                test_precision_macro,
            'test_recall_macro':
                test_recall_macro,

            'training_time_sec':
                training_time_sec,
            'peak_vram_mb':
                peak_vram_mb,
            'vram_at_load_mb':
                vram_at_load_mb,

            'trainable_params':
                trainable_params,
            'total_params':
                total_params,
            'trainable_percentage':
                trainable_percentage,

            'inference_ms':
                inference_ms,

            'lora_r':
                LORA_R,
            'lora_alpha':
                LORA_ALPHA,
            'lora_dropout':
                LORA_DROPOUT,
        }

        results_store.append(
            result
        )

        reports_store.append({
            'method': method,
            'data_level': data_level,
            'seed': seed,
            'classification_report':
                report,
        })

        save_experiment_progress()

        print(
            'LoRA result saved successfully.'
        )

        return result

    except Exception as error:
        print(
            '\nLoRA run failed | '
            f'data={data_level} | seed={seed}'
        )

        print(f'Error: {error}')

        raise

    finally:
        if trainer is not None:
            del trainer

        if model is not None:
            del model

        if base_model is not None:
            del base_model

        if 'train_dataset' in locals():
            del train_dataset

        if 'validation_dataset' in locals():
            del validation_dataset

        if 'test_dataset' in locals():
            del test_dataset

        gc.collect()
        torch.cuda.empty_cache()


print('LoRA function is ready.')

In [ ]:


import shutil
import gc


LORA_METHOD = 'lora'

LORA_DATA_LEVEL_ORDER = [
    'full_dataset',
    '25_per_class',
    '100_per_class',
]


def lora_run_completed(
    data_level,
    seed
):


    return any(
        result.get('method') == LORA_METHOD
        and result.get('data_level') == data_level
        and int(result.get('seed')) == int(seed)
        for result in all_results
    )


def delete_lora_checkpoint(
    data_level,
    seed
):

    checkpoint_dir = os.path.abspath(
        os.path.join(
            CHECKPOINT_ROOT,
            LORA_METHOD,
            data_level,
            f'seed_{seed}'
        )
    )

    checkpoint_root = os.path.abspath(
        CHECKPOINT_ROOT
    )

    if os.path.commonpath([
        checkpoint_root,
        checkpoint_dir
    ]) != checkpoint_root:
        raise ValueError(
            'Unsafe checkpoint path detected.'
        )

    if os.path.isdir(checkpoint_dir):
        shutil.rmtree(
            checkpoint_dir
        )

        print(
            'Completed checkpoint deleted: '
            f'{LORA_METHOD} | '
            f'{data_level} | seed={seed}'
        )


def save_lora_results():
    """
    Save LoRA results in a separate CSV.
    """

    lora_records = [
        result
        for result in all_results
        if result.get('method') == LORA_METHOD
    ]

    lora_results_df = pd.DataFrame(
        lora_records
    )

    lora_results_df = (
        lora_results_df
        .drop_duplicates(
            subset=[
                'method',
                'data_level',
                'seed'
            ],
            keep='last'
        )
        .reset_index(drop=True)
    )

    method_results_dir = os.path.join(
        RECOVERY_ROOT,
        'method_results'
    )

    os.makedirs(
        method_results_dir,
        exist_ok=True
    )

    results_path = os.path.join(
        method_results_dir,
        'results_lora.csv'
    )

    temporary_path = (
        results_path + '.tmp'
    )

    lora_results_df.to_csv(
        temporary_path,
        index=False
    )

    os.replace(
        temporary_path,
        results_path
    )

    return (
        lora_results_df,
        results_path
    )


TOTAL_LORA_RUNS = (
    len(LORA_DATA_LEVEL_ORDER)
    * len(SEEDS)
)

completed_count = 0


print('=' * 70)
print('LoRA EXPERIMENT')
print(
    f'Data levels : '
    f'{len(LORA_DATA_LEVEL_ORDER)}'
)
print(f'Seeds       : {len(SEEDS)}')
print(f'Total runs  : {TOTAL_LORA_RUNS}')
print('=' * 70)


for data_level in LORA_DATA_LEVEL_ORDER:

    for seed in SEEDS:

        if lora_run_completed(
            data_level=data_level,
            seed=seed
        ):
            completed_count += 1

            print(
                '\nSkipping completed run: '
                f'{LORA_METHOD} | '
                f'{data_level} | seed={seed}'
            )

            delete_lora_checkpoint(
                data_level=data_level,
                seed=seed
            )

            continue

        print(
            f'\nStarting LoRA run '
            f'{completed_count + 1}/'
            f'{TOTAL_LORA_RUNS}: '
            f'{data_level} | seed={seed}'
        )

        run_lora(
            data_level=data_level,
            seed=seed,
            results_store=all_results,
            epoch_metrics_store=all_epoch_metrics,
            reports_store=all_classification_reports
        )

        delete_lora_checkpoint(
            data_level=data_level,
            seed=seed
        )

        completed_count += 1

        lora_results_df, lora_results_path = (
            save_lora_results()
        )

        print(
            f'LoRA progress: '
            f'{completed_count}/'
            f'{TOTAL_LORA_RUNS}'
        )

        gc.collect()
        torch.cuda.empty_cache()


lora_results_df, lora_results_path = (
    save_lora_results()
)


if len(lora_results_df) != TOTAL_LORA_RUNS:
    print(
        '\nLoRA is not complete yet: '
        f'{len(lora_results_df)}/'
        f'{TOTAL_LORA_RUNS} runs.'
    )

else:
    print('\n' + '=' * 70)
    print('ALL LoRA RUNS COMPLETED')
    print(
        f'Completed runs: '
        f'{len(lora_results_df)}/'
        f'{TOTAL_LORA_RUNS}'
    )
    print(
        f'Results saved to:\n'
        f'{lora_results_path}'
    )
    print('=' * 70)

    display(
        lora_results_df[
            [
                'data_level',
                'seed',
                'test_accuracy',
                'test_f1_macro',
                'test_precision_macro',
                'test_recall_macro',
                'training_time_sec',
                'peak_vram_mb',
                'inference_ms',
            ]
        ]
    )

In [ ]:


LORA_RESULTS_PATH = os.path.join(
    RECOVERY_ROOT,
    'method_results',
    'results_lora.csv'
)

if not os.path.exists(
    LORA_RESULTS_PATH
):
    raise FileNotFoundError(
        'LoRA results were not found.'
    )


lora_results = pd.read_csv(
    LORA_RESULTS_PATH
)


lora_results = (
    lora_results
    .drop_duplicates(
        subset=[
            'method',
            'data_level',
            'seed'
        ],
        keep='last'
    )
    .reset_index(drop=True)
)

expected_lora_runs = (
    len(DATA_LEVELS)
    * len(SEEDS)
)

if len(lora_results) != expected_lora_runs:
    raise ValueError(
        f'Expected {expected_lora_runs} LoRA runs, '
        f'but found {len(lora_results)}.'
    )


for data_level in DATA_LEVELS:

    completed_seeds = set(
        lora_results.loc[
            lora_results['data_level']
            == data_level,
            'seed'
        ].astype(int)
    )

    if completed_seeds != set(SEEDS):
        raise ValueError(
            f'Incorrect Seeds for {data_level}. '
            f'Found: {sorted(completed_seeds)}'
        )



lora_per_seed_results = (
    lora_results[
        [
            'data_level',
            'seed',
            'test_accuracy',
            'test_f1_macro',
            'test_precision_macro',
            'test_recall_macro',
            'training_time_sec',
            'peak_vram_mb',
            'inference_ms',
        ]
    ]
    .sort_values(
        by=[
            'data_level',
            'seed'
        ]
    )
    .reset_index(drop=True)
)

print('\nLoRA — INDIVIDUAL SEED RESULTS')

display(
    lora_per_seed_results
)


lora_summary_rows = []

for data_level in [
    'full_dataset',
    '25_per_class',
    '100_per_class',
]:

    condition = lora_results[
        lora_results['data_level']
        == data_level
    ]

    lora_summary_rows.append({
        'Method':
            'LoRA',

        'Data Level':
            data_level,

        'N Train':
            int(
                condition[
                    'n_train_samples'
                ].iloc[0]
            ),

        'Seeds':
            len(condition),

        'Accuracy':
            format_mean_sd_ci(
                condition['test_accuracy'],
                decimals=4
            ),

        'Macro-F1':
            format_mean_sd_ci(
                condition['test_f1_macro'],
                decimals=4
            ),

        'Precision':
            format_mean_sd_ci(
                condition[
                    'test_precision_macro'
                ],
                decimals=4
            ),

        'Recall':
            format_mean_sd_ci(
                condition[
                    'test_recall_macro'
                ],
                decimals=4
            ),

        'Training Time (s)':
            format_mean_sd_ci(
                condition[
                    'training_time_sec'
                ],
                decimals=2
            ),

        'Peak VRAM (MB)':
            format_mean_sd_ci(
                condition[
                    'peak_vram_mb'
                ],
                decimals=2
            ),

        'Inference (ms/sample)':
            format_mean_sd_ci(
                condition[
                    'inference_ms'
                ],
                decimals=3
            ),

        'Trainable Parameters':
            int(
                condition[
                    'trainable_params'
                ].iloc[0]
            ),

        'Total Parameters':
            int(
                condition[
                    'total_params'
                ].iloc[0]
            ),

        'Trainable %':
            float(
                condition[
                    'trainable_percentage'
                ].iloc[0]
            ),

        'LoRA Rank':
            int(
                condition[
                    'lora_r'
                ].iloc[0]
            ),

        'LoRA Alpha':
            int(
                condition[
                    'lora_alpha'
                ].iloc[0]
            ),

        'LoRA Dropout':
            float(
                condition[
                    'lora_dropout'
                ].iloc[0]
            ),
    })


lora_summary = pd.DataFrame(
    lora_summary_rows
)


print('\n' + '=' * 140)
print('LoRA — STATISTICAL SUMMARY')
print(
    'Mean ± Sample SD '
    '[Student-t 95% Confidence Interval]'
)
print(f'Seeds per data level: {len(SEEDS)}')
print('=' * 140)

display(
    lora_summary
)


LORA_SUMMARY_PATH = os.path.join(
    RECOVERY_ROOT,
    'method_results',
    'summary_lora.csv'
)

lora_summary.to_csv(
    LORA_SUMMARY_PATH,
    index=False
)

print(
    'LoRA summary saved to:\n'
    f'{LORA_SUMMARY_PATH}'
)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type=QUANT_TYPE,
    bnb_4bit_use_double_quant=DOUBLE_QUANT,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)
print('BitsAndBytesConfig ready:')
print(f'  quant_type   : {QUANT_TYPE.upper()}')
print(f'  double_quant : {DOUBLE_QUANT}')
print(f'  compute_dtype: {COMPUTE_DTYPE}')

In [ ]:


def run_qlora(
    data_level,
    seed,
    results_store,
    epoch_metrics_store,
    reports_store
):


    method = 'qlora'

    trainer = None
    model = None
    base_model = None

    checkpoint_dir = os.path.join(
        CHECKPOINT_ROOT,
        method,
        data_level,
        f'seed_{seed}'
    )

    os.makedirs(
        checkpoint_dir,
        exist_ok=True
    )

    try:
        set_seed(seed)

        train_data = get_training_data(
            data_level=data_level,
            seed=seed
        )

        print('\n' + '=' * 70)
        print('QLoRA')
        print(f'Data level : {data_level}')
        print(f'Seed       : {seed}')
        print(f'N train    : {len(train_data)}')
        print('=' * 70)

        train_dataset = ArabicSentimentDataset(
            train_data,
            tokenizer,
            MAX_SEQ_LENGTH
        )

        validation_dataset = ArabicSentimentDataset(
            val_df,
            tokenizer,
            MAX_SEQ_LENGTH
        )

        test_dataset = ArabicSentimentDataset(
            test_df,
            tokenizer,
            MAX_SEQ_LENGTH
        )

        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

        print(
            'Loading the 4-bit quantized '
            'CAMeLBERT model...'
        )

        base_model = (
            AutoModelForSequenceClassification
            .from_pretrained(
                MODEL_NAME,
                num_labels=NUM_LABELS,
                quantization_config=bnb_config,
                device_map='auto'
            )
        )

        base_model.config.use_cache = False


        if isinstance(
            base_model.classifier,
            bnb.nn.Linear4bit
        ):
            print(
                'Replacing the quantized classification '
                'head with a trainable linear layer...'
            )

            in_features = (
                base_model.classifier.in_features
            )

            out_features = (
                base_model.classifier.out_features
            )

            new_classifier = torch.nn.Linear(
                in_features=in_features,
                out_features=out_features,
                bias=True
            )

            # Use CAMeLBERT's standard initialization
            if hasattr(
                base_model,
                '_init_weights'
            ):
                base_model._init_weights(
                    new_classifier
                )

            else:
                torch.nn.init.normal_(
                    new_classifier.weight,
                    mean=0.0,
                    std=(
                        base_model
                        .config
                        .initializer_range
                    )
                )

                torch.nn.init.zeros_(
                    new_classifier.bias
                )

            base_model.classifier = (
                new_classifier.to(
                    device='cuda',
                    dtype=COMPUTE_DTYPE
                )
            )

        else:
            base_model.classifier = (
                base_model.classifier.to(
                    device='cuda',
                    dtype=COMPUTE_DTYPE
                )
            )

        print(
            'Preparing the model for '
            '4-bit training...'
        )

        base_model = (
            prepare_model_for_kbit_training(
                base_model,
                use_gradient_checkpointing=False
            )
        )

        print(
            'Applying LoRA adapters to '
            'the quantized backbone...'
        )

        qlora_config = LoraConfig(
            task_type=TaskType.SEQ_CLS,

            r=LORA_R,
            lora_alpha=LORA_ALPHA,
            lora_dropout=LORA_DROPOUT,

            target_modules=
                LORA_TARGET_MODULES,

            modules_to_save=[
                'classifier'
            ],

            bias='none',
            inference_mode=False
        )

        model = get_peft_model(
            base_model,
            qlora_config
        )

        model.print_trainable_parameters()

        vram_at_load_mb = (
            torch.cuda.memory_allocated()
            / (1024 ** 2)
        )

        (
            trainable_params,
            total_params,
            trainable_percentage
        ) = count_trainable_params(
            model
        )

        print(
            f'Trainable parameters: '
            f'{trainable_params:,} / '
            f'{total_params:,} '
            f'({trainable_percentage:.4f}%)'
        )

        print(
            f'VRAM after model loading: '
            f'{vram_at_load_mb:.2f} MB'
        )

        time_callback = (
            TrainingTimeCallback()
        )

        epoch_callback = EpochMetricsCallback(
            method=method,
            data_level=data_level,
            seed=seed
        )

        trainer = Trainer(
            model=model,

            args=make_training_args(
                checkpoint_dir=checkpoint_dir,
                seed=seed,
                is_qlora=True
            ),

            train_dataset=train_dataset,
            eval_dataset=validation_dataset,

            compute_metrics=compute_metrics,

            callbacks=[
                time_callback,
                epoch_callback
            ]
        )

        last_checkpoint = get_last_checkpoint(
            checkpoint_dir
        )

        if last_checkpoint is not None:
            print(
                'Checkpoint found. Resuming from:\n'
                f'{last_checkpoint}'
            )

            trainer.train(
                resume_from_checkpoint=
                    last_checkpoint
            )

        else:
            print(
                'No checkpoint found. '
                'Starting from the beginning.'
            )

            trainer.train()

        training_time_sec = (
            time_callback.total_train_time_sec
        )

        peak_vram_mb = (
            torch.cuda.max_memory_allocated()
            / (1024 ** 2)
        )

        epoch_metrics_store.extend(
            collect_epoch_metrics_from_trainer(
                trainer=trainer,
                method=method,
                data_level=data_level,
                seed=seed
            )
        )

        (
            test_accuracy,
            test_f1_macro,
            test_precision_macro,
            test_recall_macro,
            report
        ) = evaluate_on_test(
            trainer,
            test_dataset
        )

        inference_ms = measure_inference_time(
            model=trainer.model,
            tokenizer=tokenizer,
            texts=test_df['review'].tolist(),
            max_length=MAX_SEQ_LENGTH
        )

        print('\nRun summary')
        print('-' * 70)
        print(
            f'Test Accuracy  : '
            f'{test_accuracy:.4f}'
        )
        print(
            f'Test Macro-F1  : '
            f'{test_f1_macro:.4f}'
        )
        print(
            f'Test Precision : '
            f'{test_precision_macro:.4f}'
        )
        print(
            f'Test Recall    : '
            f'{test_recall_macro:.4f}'
        )
        print(
            f'Training time  : '
            f'{training_time_sec:.2f} seconds'
        )
        print(
            f'Peak VRAM      : '
            f'{peak_vram_mb:.2f} MB'
        )
        print(
            f'Inference      : '
            f'{inference_ms:.3f} ms/sample'
        )
        print(
            'Test model     : '
            'final training epoch'
        )

        print(
            f'\nClassification report:\n'
            f'{report}'
        )

        result = {
            'method': method,
            'data_level': data_level,
            'seed': seed,
            'n_train_samples': len(train_data),

            'test_accuracy':
                test_accuracy,
            'test_f1_macro':
                test_f1_macro,
            'test_precision_macro':
                test_precision_macro,
            'test_recall_macro':
                test_recall_macro,

            'training_time_sec':
                training_time_sec,
            'peak_vram_mb':
                peak_vram_mb,
            'vram_at_load_mb':
                vram_at_load_mb,

            'trainable_params':
                trainable_params,
            'total_params':
                total_params,
            'trainable_percentage':
                trainable_percentage,

            'inference_ms':
                inference_ms,

            'lora_r':
                LORA_R,
            'lora_alpha':
                LORA_ALPHA,
            'lora_dropout':
                LORA_DROPOUT,

            'quantization_bits':
                4,
            'quant_type':
                QUANT_TYPE,
            'double_quant':
                DOUBLE_QUANT,
        }

        results_store.append(
            result
        )

        reports_store.append({
            'method': method,
            'data_level': data_level,
            'seed': seed,
            'classification_report':
                report,
        })

        save_experiment_progress()

        print(
            'QLoRA result saved successfully.'
        )

        return result

    except Exception as error:
        print(
            '\nQLoRA run failed | '
            f'data={data_level} | seed={seed}'
        )

        print(f'Error: {error}')

        raise

    finally:
        if trainer is not None:
            del trainer

        if model is not None:
            del model

        if base_model is not None:
            del base_model

        if 'train_dataset' in locals():
            del train_dataset

        if 'validation_dataset' in locals():
            del validation_dataset

        if 'test_dataset' in locals():
            del test_dataset

        gc.collect()
        torch.cuda.empty_cache()


print('QLoRA function is ready.')


In [ ]:
QLORA_METHOD = 'qlora'

QLORA_DATA_LEVEL_ORDER = [
    'full_dataset',
    '25_per_class',
    '100_per_class',
]


def qlora_run_completed(
    data_level,
    seed
):


    return any(
        result.get('method') == QLORA_METHOD
        and result.get('data_level') == data_level
        and int(result.get('seed')) == int(seed)
        for result in all_results
    )


def delete_qlora_checkpoint(
    data_level,
    seed
):


    checkpoint_dir = os.path.abspath(
        os.path.join(
            CHECKPOINT_ROOT,
            QLORA_METHOD,
            data_level,
            f'seed_{seed}'
        )
    )

    checkpoint_root = os.path.abspath(
        CHECKPOINT_ROOT
    )

    if os.path.commonpath([
        checkpoint_root,
        checkpoint_dir
    ]) != checkpoint_root:
        raise ValueError(
            'Unsafe checkpoint path detected.'
        )

    if os.path.isdir(checkpoint_dir):
        shutil.rmtree(
            checkpoint_dir
        )

        print(
            'Completed checkpoint deleted: '
            f'{QLORA_METHOD} | '
            f'{data_level} | seed={seed}'
        )


def save_qlora_results():
    """
    Save QLoRA results in a separate CSV.
    """

    qlora_records = [
        result
        for result in all_results
        if result.get('method') == QLORA_METHOD
    ]

    qlora_results_df = pd.DataFrame(
        qlora_records
    )

    qlora_results_df = (
        qlora_results_df
        .drop_duplicates(
            subset=[
                'method',
                'data_level',
                'seed'
            ],
            keep='last'
        )
        .reset_index(drop=True)
    )

    method_results_dir = os.path.join(
        RECOVERY_ROOT,
        'method_results'
    )

    os.makedirs(
        method_results_dir,
        exist_ok=True
    )

    results_path = os.path.join(
        method_results_dir,
        'results_qlora.csv'
    )

    temporary_path = (
        results_path + '.tmp'
    )

    qlora_results_df.to_csv(
        temporary_path,
        index=False
    )

    os.replace(
        temporary_path,
        results_path
    )

    return (
        qlora_results_df,
        results_path
    )


TOTAL_QLORA_RUNS = (
    len(QLORA_DATA_LEVEL_ORDER)
    * len(SEEDS)
)

completed_count = 0


print('=' * 70)
print('QLoRA EXPERIMENT')
print(
    f'Data levels : '
    f'{len(QLORA_DATA_LEVEL_ORDER)}'
)
print(f'Seeds       : {len(SEEDS)}')
print(f'Total runs  : {TOTAL_QLORA_RUNS}')
print('=' * 70)


for data_level in QLORA_DATA_LEVEL_ORDER:

    for seed in SEEDS:

        if qlora_run_completed(
            data_level=data_level,
            seed=seed
        ):
            completed_count += 1

            print(
                '\nSkipping completed run: '
                f'{QLORA_METHOD} | '
                f'{data_level} | seed={seed}'
            )

            delete_qlora_checkpoint(
                data_level=data_level,
                seed=seed
            )

            continue

        print(
            f'\nStarting QLoRA run '
            f'{completed_count + 1}/'
            f'{TOTAL_QLORA_RUNS}: '
            f'{data_level} | seed={seed}'
        )

        run_qlora(
            data_level=data_level,
            seed=seed,
            results_store=all_results,
            epoch_metrics_store=all_epoch_metrics,
            reports_store=all_classification_reports
        )

        delete_qlora_checkpoint(
            data_level=data_level,
            seed=seed
        )

        completed_count += 1

        qlora_results_df, qlora_results_path = (
            save_qlora_results()
        )

        print(
            f'QLoRA progress: '
            f'{completed_count}/'
            f'{TOTAL_QLORA_RUNS}'
        )

        gc.collect()
        torch.cuda.empty_cache()


qlora_results_df, qlora_results_path = (
    save_qlora_results()
)


if len(qlora_results_df) != TOTAL_QLORA_RUNS:
    print(
        '\nQLoRA is not complete yet: '
        f'{len(qlora_results_df)}/'
        f'{TOTAL_QLORA_RUNS} runs.'
    )

else:
    print('\n' + '=' * 70)
    print('ALL QLoRA RUNS COMPLETED')
    print(
        f'Completed runs: '
        f'{len(qlora_results_df)}/'
        f'{TOTAL_QLORA_RUNS}'
    )
    print(
        f'Results saved to:\n'
        f'{qlora_results_path}'
    )
    print('=' * 70)

    display(
        qlora_results_df[
            [
                'data_level',
                'seed',
                'test_accuracy',
                'test_f1_macro',
                'test_precision_macro',
                'test_recall_macro',
                'training_time_sec',
                'peak_vram_mb',
                'inference_ms',
            ]
        ]
    )

In [ ]:

QLORA_RESULTS_PATH = os.path.join(
    RECOVERY_ROOT,
    'method_results',
    'results_qlora.csv'
)

if not os.path.exists(
    QLORA_RESULTS_PATH
):
    raise FileNotFoundError(
        'QLoRA results were not found.'
    )


qlora_results = pd.read_csv(
    QLORA_RESULTS_PATH
)


qlora_results = (
    qlora_results
    .drop_duplicates(
        subset=[
            'method',
            'data_level',
            'seed'
        ],
        keep='last'
    )
    .reset_index(drop=True)
)


QLORA_TOTAL_PARAMETERS = 109_674_244

qlora_results[
    'total_params'
] = QLORA_TOTAL_PARAMETERS

qlora_results[
    'trainable_percentage'
] = (
    qlora_results[
        'trainable_params'
    ]
    / QLORA_TOTAL_PARAMETERS
    * 100
)


temporary_qlora_path = (
    QLORA_RESULTS_PATH + '.tmp'
)

qlora_results.to_csv(
    temporary_qlora_path,
    index=False
)

os.replace(
    temporary_qlora_path,
    QLORA_RESULTS_PATH
)



expected_qlora_runs = (
    len(DATA_LEVELS)
    * len(SEEDS)
)

if len(qlora_results) != expected_qlora_runs:
    raise ValueError(
        f'Expected {expected_qlora_runs} QLoRA runs, '
        f'but found {len(qlora_results)}.'
    )


for data_level in DATA_LEVELS:

    completed_seeds = set(
        qlora_results.loc[
            qlora_results['data_level']
            == data_level,
            'seed'
        ].astype(int)
    )

    if completed_seeds != set(SEEDS):
        raise ValueError(
            f'Incorrect Seeds for {data_level}. '
            f'Found: {sorted(completed_seeds)}'
        )

qlora_per_seed_results = (
    qlora_results[
        [
            'data_level',
            'seed',
            'test_accuracy',
            'test_f1_macro',
            'test_precision_macro',
            'test_recall_macro',
            'training_time_sec',
            'peak_vram_mb',
            'inference_ms',
            'trainable_params',
            'total_params',
            'trainable_percentage',
        ]
    ]
    .sort_values(
        by=[
            'data_level',
            'seed'
        ]
    )
    .reset_index(drop=True)
)

print('\nQLoRA — INDIVIDUAL SEED RESULTS')

display(
    qlora_per_seed_results
)

qlora_summary_rows = []

for data_level in [
    'full_dataset',
    '25_per_class',
    '100_per_class',
]:

    condition = qlora_results[
        qlora_results['data_level']
        == data_level
    ]

    qlora_summary_rows.append({
        'Method':
            'QLoRA',

        'Data Level':
            data_level,

        'N Train':
            int(
                condition[
                    'n_train_samples'
                ].iloc[0]
            ),

        'Seeds':
            len(condition),

        'Accuracy':
            format_mean_sd_ci(
                condition['test_accuracy'],
                decimals=4
            ),

        'Macro-F1':
            format_mean_sd_ci(
                condition['test_f1_macro'],
                decimals=4
            ),

        'Precision':
            format_mean_sd_ci(
                condition[
                    'test_precision_macro'
                ],
                decimals=4
            ),

        'Recall':
            format_mean_sd_ci(
                condition[
                    'test_recall_macro'
                ],
                decimals=4
            ),

        'Training Time (s)':
            format_mean_sd_ci(
                condition[
                    'training_time_sec'
                ],
                decimals=2
            ),

        'Peak VRAM (MB)':
            format_mean_sd_ci(
                condition[
                    'peak_vram_mb'
                ],
                decimals=2
            ),

        'Inference (ms/sample)':
            format_mean_sd_ci(
                condition[
                    'inference_ms'
                ],
                decimals=3
            ),

        'Trainable Parameters':
            int(
                condition[
                    'trainable_params'
                ].iloc[0]
            ),

        'Total Parameters':
            QLORA_TOTAL_PARAMETERS,

        'Trainable %':
            float(
                condition[
                    'trainable_percentage'
                ].iloc[0]
            ),

        'LoRA Rank':
            int(
                condition[
                    'lora_r'
                ].iloc[0]
            ),

        'LoRA Alpha':
            int(
                condition[
                    'lora_alpha'
                ].iloc[0]
            ),

        'LoRA Dropout':
            float(
                condition[
                    'lora_dropout'
                ].iloc[0]
            ),

        'Quantization':
            '4-bit NF4',

        'Double Quantization':
            bool(
                condition[
                    'double_quant'
                ].iloc[0]
            ),
    })


qlora_summary = pd.DataFrame(
    qlora_summary_rows
)




print('\n' + '=' * 140)
print('QLoRA — STATISTICAL SUMMARY')
print(
    'Mean ± Sample SD '
    '[Student-t 95% Confidence Interval]'
)
print(f'Seeds per data level: {len(SEEDS)}')
print('=' * 140)

display(
    qlora_summary
)


QLORA_SUMMARY_PATH = os.path.join(
    RECOVERY_ROOT,
    'method_results',
    'summary_qlora.csv'
)

qlora_summary.to_csv(
    QLORA_SUMMARY_PATH,
    index=False
)

print(
    'QLoRA summary saved to:\n'
    f'{QLORA_SUMMARY_PATH}'
)

In [ ]:


METHOD_RESULT_FILES = {
    'full_finetuning':
        'results_full_finetuning.csv',

    'frozen_backbone':
        'results_frozen_backbone.csv',

    'lora':
        'results_lora.csv',

    'qlora':
        'results_qlora.csv',
}


METHOD_DISPLAY_NAMES = {
    'full_finetuning':
        'Full Fine-Tuning',

    'frozen_backbone':
        'Frozen Backbone',

    'lora':
        'LoRA',

    'qlora':
        'QLoRA',
}


METHOD_ORDER = [
    'full_finetuning',
    'frozen_backbone',
    'lora',
    'qlora',
]


DATA_LEVEL_ORDER = [
    'full_dataset',
    '25_per_class',
    '100_per_class',
]


METHOD_RESULTS_DIR = os.path.join(
    RECOVERY_ROOT,
    'method_results'
)



method_dataframes = []

for method, filename in METHOD_RESULT_FILES.items():

    file_path = os.path.join(
        METHOD_RESULTS_DIR,
        filename
    )

    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f'Results file missing for {method}:\n'
            f'{file_path}'
        )

    method_df = pd.read_csv(
        file_path
    )

    method_df['method'] = method

    method_dataframes.append(
        method_df
    )

    print(
        f'Loaded {method}: '
        f'{len(method_df)} runs'
    )


combined_results = pd.concat(
    method_dataframes,
    ignore_index=True
)



combined_results = (
    combined_results
    .drop_duplicates(
        subset=[
            'method',
            'data_level',
            'seed'
        ],
        keep='last'
    )
    .reset_index(drop=True)
)

combined_results['seed'] = (
    combined_results['seed']
    .astype(int)
)



QLORA_TOTAL_PARAMETERS = 109_674_244

qlora_mask = (
    combined_results['method']
    == 'qlora'
)

combined_results.loc[
    qlora_mask,
    'total_params'
] = QLORA_TOTAL_PARAMETERS

combined_results.loc[
    qlora_mask,
    'trainable_percentage'
] = (
    combined_results.loc[
        qlora_mask,
        'trainable_params'
    ]
    / QLORA_TOTAL_PARAMETERS
    * 100
)


EXPECTED_TOTAL_RUNS = (
    len(METHOD_ORDER)
    * len(DATA_LEVEL_ORDER)
    * len(SEEDS)
)

if len(combined_results) != EXPECTED_TOTAL_RUNS:
    raise ValueError(
        f'Expected {EXPECTED_TOTAL_RUNS} unique runs, '
        f'but found {len(combined_results)}.'
    )


for method in METHOD_ORDER:

    for data_level in DATA_LEVEL_ORDER:

        condition_seeds = set(
            combined_results.loc[
                (
                    combined_results['method']
                    == method
                )
                & (
                    combined_results['data_level']
                    == data_level
                ),
                'seed'
            ]
        )

        if condition_seeds != set(SEEDS):
            raise ValueError(
                f'Incorrect Seeds for '
                f'{method} | {data_level}. '
                f'Found: {sorted(condition_seeds)}'
            )


print(
    f'Validation passed: '
    f'{len(combined_results)}/'
    f'{EXPECTED_TOTAL_RUNS} unique runs.'
)



ALL_RUNS_PATH = os.path.join(
    METHOD_RESULTS_DIR,
    'all_60_runs.csv'
)

combined_results.to_csv(
    ALL_RUNS_PATH,
    index=False
)

print(
    'All individual Seed results saved to:\n'
    f'{ALL_RUNS_PATH}'
)


all_seed_results = (
    combined_results[
        [
            'method',
            'data_level',
            'seed',
            'n_train_samples',
            'test_accuracy',
            'test_f1_macro',
            'test_precision_macro',
            'test_recall_macro',
            'training_time_sec',
            'peak_vram_mb',
            'inference_ms',
        ]
    ]
    .copy()
)

all_seed_results['method'] = (
    pd.Categorical(
        all_seed_results['method'],
        categories=METHOD_ORDER,
        ordered=True
    )
)

all_seed_results['data_level'] = (
    pd.Categorical(
        all_seed_results['data_level'],
        categories=DATA_LEVEL_ORDER,
        ordered=True
    )
)

all_seed_results = (
    all_seed_results
    .sort_values(
        by=[
            'method',
            'data_level',
            'seed'
        ]
    )
    .reset_index(drop=True)
)


print('\nALL INDIVIDUAL SEED RESULTS')

display(
    all_seed_results
)

METRICS = {
    'test_accuracy':
        'accuracy',

    'test_f1_macro':
        'macro_f1',

    'test_precision_macro':
        'precision',

    'test_recall_macro':
        'recall',

    'training_time_sec':
        'training_time_sec',

    'peak_vram_mb':
        'peak_vram_mb',

    'inference_ms':
        'inference_ms',
}


numeric_summary_rows = []
formatted_summary_rows = []


for method in METHOD_ORDER:

    for data_level in DATA_LEVEL_ORDER:

        condition = combined_results[
            (
                combined_results['method']
                == method
            )
            & (
                combined_results['data_level']
                == data_level
            )
        ]

        numeric_row = {
            'method': method,
            'method_display':
                METHOD_DISPLAY_NAMES[method],
            'data_level': data_level,
            'n_train_samples':
                int(
                    condition[
                        'n_train_samples'
                    ].iloc[0]
                ),
            'seeds': len(condition),
            'trainable_params':
                int(
                    condition[
                        'trainable_params'
                    ].iloc[0]
                ),
            'total_params':
                int(
                    condition[
                        'total_params'
                    ].iloc[0]
                ),
            'trainable_percentage':
                float(
                    condition[
                        'trainable_percentage'
                    ].iloc[0]
                ),
        }

        for source_column, output_name in METRICS.items():

            statistics = calculate_mean_sd_ci(
                values=condition[source_column],
                confidence=0.95
            )

            numeric_row[
                f'{output_name}_mean'
            ] = statistics['mean']

            numeric_row[
                f'{output_name}_sd'
            ] = statistics['sd']

            numeric_row[
                f'{output_name}_ci_lower'
            ] = statistics['ci_lower']

            numeric_row[
                f'{output_name}_ci_upper'
            ] = statistics['ci_upper']

        numeric_summary_rows.append(
            numeric_row
        )

        formatted_summary_rows.append({
            'Method':
                METHOD_DISPLAY_NAMES[method],

            'Data Level':
                data_level,

            'N Train':
                int(
                    condition[
                        'n_train_samples'
                    ].iloc[0]
                ),

            'Seeds':
                len(condition),

            'Accuracy':
                format_mean_sd_ci(
                    condition['test_accuracy'],
                    decimals=4
                ),

            'Macro-F1':
                format_mean_sd_ci(
                    condition['test_f1_macro'],
                    decimals=4
                ),

            'Precision':
                format_mean_sd_ci(
                    condition[
                        'test_precision_macro'
                    ],
                    decimals=4
                ),

            'Recall':
                format_mean_sd_ci(
                    condition[
                        'test_recall_macro'
                    ],
                    decimals=4
                ),

            'Training Time (s)':
                format_mean_sd_ci(
                    condition[
                        'training_time_sec'
                    ],
                    decimals=2
                ),

            'Peak VRAM (MB)':
                format_mean_sd_ci(
                    condition[
                        'peak_vram_mb'
                    ],
                    decimals=2
                ),

            'Inference (ms/sample)':
                format_mean_sd_ci(
                    condition[
                        'inference_ms'
                    ],
                    decimals=3
                ),

            'Trainable Parameters':
                int(
                    condition[
                        'trainable_params'
                    ].iloc[0]
                ),

            'Total Parameters':
                int(
                    condition[
                        'total_params'
                    ].iloc[0]
                ),

            'Trainable %':
                float(
                    condition[
                        'trainable_percentage'
                    ].iloc[0]
                ),
        })


numeric_summary = pd.DataFrame(
    numeric_summary_rows
)

final_summary = pd.DataFrame(
    formatted_summary_rows
)

print('\n' + '=' * 150)
print('CAMeLBERT ADAPTATION STUDY — FINAL RESULTS')
print('4 Methods × 3 Data Levels × 5 Seeds = 60 Runs')
print(
    'Mean ± Sample SD '
    '[Student-t 95% Confidence Interval]'
)
print('=' * 150)

display(
    final_summary
)

NUMERIC_SUMMARY_PATH = os.path.join(
    METHOD_RESULTS_DIR,
    'statistical_summary_numeric.csv'
)

FINAL_SUMMARY_PATH = os.path.join(
    METHOD_RESULTS_DIR,
    'summary_all_methods.csv'
)


numeric_summary.to_csv(
    NUMERIC_SUMMARY_PATH,
    index=False
)

final_summary.to_csv(
    FINAL_SUMMARY_PATH,
    index=False
)


print(
    'Numeric statistical summary saved to:\n'
    f'{NUMERIC_SUMMARY_PATH}'
)

print(
    '\nFormatted final summary saved to:\n'
    f'{FINAL_SUMMARY_PATH}'
)

In [ ]:


import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D



REPORT_DIR = os.path.join(
    RECOVERY_ROOT,
    'figures'
)

os.makedirs(
    REPORT_DIR,
    exist_ok=True
)

EXPECTED_SEEDS = len(SEEDS)

STRATEGY_ORDER = [
    'Full Fine-Tuning',
    'Frozen Backbone',
    'LoRA',
    'QLoRA',
]

METHOD_DISPLAY_NAMES = {
    'full_finetuning': 'Full Fine-Tuning',
    'frozen_backbone': 'Frozen Backbone',
    'lora': 'LoRA',
    'qlora': 'QLoRA',
}

DATA_LEVEL_TO_LABEL = {
    '25_per_class': '25/class',
    '100_per_class': '100/class',
    'full_dataset': 'full_dataset',
}

COLORS = {
    'Full Fine-Tuning': '#2563EB',
    'Frozen Backbone': '#16A34A',
    'LoRA': '#DC2626',
    'QLoRA': '#9333EA',
}

MARKERS = {
    'Full Fine-Tuning': 'o',
    'Frozen Backbone': 's',
    'LoRA': '^',
    'QLoRA': 'D',
}



raw = combined_results.copy()

raw['strategy'] = (
    raw['method']
    .map(METHOD_DISPLAY_NAMES)
)

raw['data_label'] = (
    raw['data_level']
    .map(DATA_LEVEL_TO_LABEL)
)


agg_line = (
    raw.groupby(
        ['strategy', 'data_label'],
        sort=False,
        observed=True
    )
    .agg(
        F1_mean=('test_f1_macro', 'mean'),
        F1_std=('test_f1_macro', 'std'),
        Seeds=('seed', 'nunique'),
    )
    .reset_index()
)

x_pos = {
    '25/class': 0,
    '100/class': 1,
    'full_dataset': 2,
}

x_ticks = [0, 1, 2]

x_labels = [
    '25 / class',
    '100 / class',
    'Full Dataset'
]

fig, ax = plt.subplots(
    figsize=(7, 4.5)
)

for strategy in STRATEGY_ORDER:

    sub = agg_line[
        agg_line['strategy'] == strategy
    ].copy()

    sub['x'] = (
        sub['data_label']
        .map(x_pos)
    )

    sub = sub.sort_values('x')

    x_values = (
        sub['x']
        .astype(float)
        .to_numpy()
    )

    f1_mean = (
        sub['F1_mean']
        .astype(float)
        .to_numpy()
    )

    f1_std = (
        sub['F1_std']
        .astype(float)
        .to_numpy()
    )

    ax.plot(
        x_values,
        f1_mean,
        color=COLORS[strategy],
        marker=MARKERS[strategy],
        linewidth=2,
        markersize=7,
        label=strategy,
    )

    ax.fill_between(
        x_values,
        f1_mean - f1_std,
        f1_mean + f1_std,
        color=COLORS[strategy],
        alpha=0.12,
    )

ax.set_xticks(
    x_ticks
)

ax.set_xticklabels(
    x_labels
)

ax.set_xlabel(
    'Training-Data Level'
)

ax.set_ylabel(
    'Macro-F1'
)

ax.set_title(
    'Macro-F1 vs. Training-Data Level\n'
    f'(mean ± 1 std across {EXPECTED_SEEDS} seeds)'
)

ax.legend(
    title='Method',
    framealpha=0.9
)

ax.yaxis.set_major_formatter(
    mticker.FormatStrFormatter(
        '%.3f'
    )
)

fig.tight_layout()

fig.savefig(
    os.path.join(
        REPORT_DIR,
        'fig1_macro_f1_vs_data_level.pdf'
    ),
    bbox_inches='tight'
)

fig.savefig(
    os.path.join(
        REPORT_DIR,
        'fig1_macro_f1_vs_data_level.png'
    ),
    bbox_inches='tight',
    dpi=300
)

plt.show()



def scatter_tradeoff(
    data_level_key,
    axis
):

    subset = raw[
        raw['data_label']
        == data_level_key
    ].copy()

    aggregate = (
        subset
        .groupby(
            'strategy',
            sort=False,
            observed=True
        )
        .agg(
            F1_mean=(
                'test_f1_macro',
                'mean'
            ),
            F1_std=(
                'test_f1_macro',
                'std'
            ),
            mem_mean=(
                'peak_vram_mb',
                'mean'
            ),
            mem_std=(
                'peak_vram_mb',
                'std'
            ),
            Seeds=(
                'seed',
                'nunique'
            ),
        )
        .reset_index()
    )

    if not (
        aggregate['Seeds']
        == EXPECTED_SEEDS
    ).all():
        raise ValueError(
            f'Each method must contain '
            f'{EXPECTED_SEEDS} seeds.'
        )

    for _, row in aggregate.iterrows():

        strategy = row['strategy']

        axis.errorbar(
            row['mem_mean'],
            row['F1_mean'],

            xerr=row['mem_std'],
            yerr=row['F1_std'],

            fmt=MARKERS[strategy],
            color=COLORS[strategy],

            markersize=10,
            capsize=4,
            linewidth=1.5,
        )

        axis.annotate(
            strategy,

            xy=(
                row['mem_mean'],
                row['F1_mean']
            ),

            xytext=(0, 9),

            textcoords='offset points',
            ha='center',
            fontsize=8.5,

            color=COLORS[strategy],
        )

    axis.set_xlabel(
        'Peak GPU Memory (MB)'
    )

    axis.set_ylabel(
        'Macro-F1'
    )

    axis.yaxis.set_major_formatter(
        mticker.FormatStrFormatter(
            '%.3f'
        )
    )


fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5),
    sharey=False
)

scatter_tradeoff(
    '100/class',
    axes[0]
)

scatter_tradeoff(
    '25/class',
    axes[1]
)

axes[0].set_title(
    '(a) 100 per class'
)

axes[1].set_title(
    '(b) 25 per class'
)

legend_handles = [
    Line2D(
        [0],
        [0],

        marker=MARKERS[strategy],
        color=COLORS[strategy],

        linewidth=0,
        markersize=9,

        label=strategy
    )

    for strategy in STRATEGY_ORDER
]

fig.legend(
    handles=legend_handles,
    title='Method',

    loc='lower center',
    ncol=4,

    bbox_to_anchor=(
        0.5,
        -0.08
    ),

    framealpha=0.9,
)

fig.suptitle(
    'Performance–Cost Trade-Off: '
    'Macro-F1 vs. Peak GPU Memory\n'
    f'(mean ± 1 sample SD across {EXPECTED_SEEDS} seeds)',
    fontsize=12,
    y=1.02,
)

fig.tight_layout()

fig.savefig(
    os.path.join(
        REPORT_DIR,
        'fig2_performance_cost_tradeoff.pdf'
    ),
    bbox_inches='tight'
)

fig.savefig(
    os.path.join(
        REPORT_DIR,
        'fig2_performance_cost_tradeoff.png'
    ),
    bbox_inches='tight',
    dpi=300
)

plt.show()

print(
    'Final figures saved successfully.'
)

In [ ]:
from itertools import combinations

confidence = 0.95

METHOD_DISPLAY = {
    'full_finetuning': 'Full Fine-Tuning',
    'frozen_backbone': 'Frozen Backbone',
    'lora': 'LoRA',
    'qlora': 'QLoRA',
}

METHOD_ORDER = [
    'full_finetuning',
    'frozen_backbone',
    'lora',
    'qlora',
]

DATA_LEVEL_ORDER = [
    'full_dataset',
    '25_per_class',
    '100_per_class',
]

pairwise_rows = []

for data_level in DATA_LEVEL_ORDER:

    level = combined_results[
        combined_results["data_level"] == data_level
    ][["method", "seed", "test_f1_macro"]]

    pivot = level.pivot(
        index="seed",
        columns="method",
        values="test_f1_macro"
    )

    pivot = pivot.loc[SEEDS]

    for method_a, method_b in combinations(METHOD_ORDER, 2):

        differences = (
            pivot[method_a] -
            pivot[method_b]
        ).astype(float)

        n = len(differences)

        mean_difference = differences.mean()

        sample_sd = differences.std(ddof=1)

        standard_error = (
            sample_sd /
            np.sqrt(n)
        )

        degrees_of_freedom = n - 1

        probability = (
            1 + confidence
        ) / 2

        t_critical = t.ppf(
            probability,
            degrees_of_freedom
        )

        margin_of_error = (
            t_critical *
            standard_error
        )

        ci_lower = (
            mean_difference -
            margin_of_error
        )

        ci_upper = (
            mean_difference +
            margin_of_error
        )

        pairwise_rows.append({

            "Data Level": data_level,

            "Comparison":
                f"{METHOD_DISPLAY[method_a]} vs "
                f"{METHOD_DISPLAY[method_b]}",

            "Mean Difference":
                round(mean_difference, 4),

            "95% CI":
                f"[{ci_lower:.4f}, {ci_upper:.4f}]"

        })

pairwise_results = pd.DataFrame(pairwise_rows)

display(pairwise_results)